In [3]:
# imports
import numpy as np
import pandas as pd

from collections import defaultdict
from operator import itemgetter
from os import path

from common.county_geometry import central_angle
from common.county_geometry import earth_radius_mi


In [4]:
# bring in intersections -> DataFrame intersections 
intersection_data_path = "./data/cleaner/intersections_clean.json"

grid_column = "KY_grid_XY"
# I've changed this a few times and it's getting annoying to change it everytime :)

# import intersection data
def read_in_intersections(path_to_intersection_data):
    df = pd.read_json(path_to_intersection_data, orient='records', lines=True)
    df['GEOMETRY'] = df.GEOMETRY.apply(np.array)
    df[grid_column] = df[grid_column].apply(np.array)
    return df.set_index("INTID")       

intersections = read_in_intersections(intersection_data_path).drop([grid_column], axis=1)

#intersections.GEOMETRY = intersections.GEOMETRY.apply(np.array)
#intersections.GEOMETRY.iloc[0] # == array([-85.51043903,  38.20587931]) # good 

intersections.head()

# from common.county_geometry import convert_points

# grid_to_LL = intersections.KY_grid_XY.apply(convert_points.point_to_ll)
# LL_to_grid = intersections.GEOMETRY.apply(convert_points.point_to_grid)



,FST_ROADNAME,SEC_ROADNAME,SIFCODE1,SIFCODE2,FST_SIFID,SEC_SIFID,GEOMETRY
INTID,,,,,,,
5710837346,REHL RD,W REHL CT,5464,7662,4976,6856,"[-85.51044384084946, 38.20588660809318]"
10005800273,REHL RD,TUCKER STATION RD,5464,6551,4976,5908,"[-85.52816882852979, 38.20037561255241]"
14300767569,REHL RD,TUCKER STATION RD,5464,6551,4976,5908,"[-85.52841872335158, 38.200360349057064]"
18011691414,I 64 EAST,I 265 RAMP,3194,9996,3076,8763,"[-85.50495414554669, 38.22260344055154]"
23945910678,I 265 NORTH,I 265 RAMP,9349,9996,8197,8763,"[-85.50554642815675, 38.222127288727606]"


In [5]:
# bring in centerlines -> DataFrame centerlines
centerlines_path = "./data/cleaner/centerlines_clean.json"

def read_in_centerlines(path_to_data):
    df = pd.read_json(path_to_data, orient='records', lines=True)
    return df.set_index("OBJECTID")

centerline_data = read_in_centerlines(centerlines_path)
#centerlines.head()

# fix geometry column

def get_geo_ends(df):
    ends = df.GEOMETRY.transform({"GEOLOW":itemgetter(0), 'GEOHI':itemgetter(-1)})
    df['GEO_start'] = ends.GEOLOW.apply(np.array)
    df['GEO_end'] = ends.GEOHI.apply(np.array) 
    return df

centerlines = get_geo_ends(centerline_data)

# set aside full geo for now:
centerline_GEOMETRY = centerlines.GEOMETRY
centerlines = centerlines.drop(["GEOMETRY"], axis=1)

display(centerline_GEOMETRY.head())
centerlines.head()


OBJECTID
1    [[-85.6809503218278, 38.158867088749105], [-85...
2    [[-85.80120122370631, 38.23056379303306], [-85...
3    [[-85.80501498815028, 38.228933021550915], [-8...
4    [[-85.68020545343364, 38.24806713661984], [-85...
5    [[-85.74203979766622, 38.211814770925905], [-8...
Name: GEOMETRY, dtype: object

,ROADNAME,SIFID,SIFCODE,low_cross_ROADNAME,SIFIDLOW,LOCROSSSIF,hi_cross_ROADNAME,SIFIDHI,HICROSSSIF,CORE_CLASS,GEO_start,GEO_end
OBJECTID,,,,,,,,,,,,
1,SERENITY CT,8665,9887,DELLAFAY DR,1550,1557,DEAD END,8594,9811,LOCAL,"[-85.6809503218278, 38.158867088749105]","[-85.68123463490603, 38.158180484404916]"
2,S 28TH ST,5926,6570,W HILL ST,2854,2934,W GAULBERT AVE,2470,2499,LOCAL,"[-85.80120122370631, 38.23056379303306]","[-85.8013692697861, 38.22934335950893]"
3,BEECH ST,473,0458,WILSON AVE,6487,7212,DR WILLIAM G WEATHERS DR,10596,D596,LOCAL,"[-85.80501498815028, 38.228933021550915]","[-85.80483162492965, 38.22753788732054]"
4,GARDEN DR,2442,2470,RAINBOW DR,13394,5391,POPPY WAY,4702,5128,PRIMARY COLLECTOR,"[-85.68020545343364, 38.24806713661984]","[-85.67986300781635, 38.24760815991686]"
5,PARKWAY DR,4573,4974,MOUNT CLAIRE AVE,4162,4476,DEAD END,8594,9811,LOCAL,"[-85.74203979766622, 38.211814770925905]","[-85.74164752771065, 38.21196859682181]"


In [6]:
# remove interstates from centerlines ?\
# - and other roads like ramps?
exclusions = ('EXPRESSWAY', 'INTERSTATE RAMP')
centerlines = centerlines[~centerlines.CORE_CLASS.isin(exclusions)]

# remove these from intersections as well? 

centerlines.head()

,ROADNAME,SIFID,SIFCODE,low_cross_ROADNAME,SIFIDLOW,LOCROSSSIF,hi_cross_ROADNAME,SIFIDHI,HICROSSSIF,CORE_CLASS,GEO_start,GEO_end
OBJECTID,,,,,,,,,,,,
1,SERENITY CT,8665,9887,DELLAFAY DR,1550,1557,DEAD END,8594,9811,LOCAL,"[-85.6809503218278, 38.158867088749105]","[-85.68123463490603, 38.158180484404916]"
2,S 28TH ST,5926,6570,W HILL ST,2854,2934,W GAULBERT AVE,2470,2499,LOCAL,"[-85.80120122370631, 38.23056379303306]","[-85.8013692697861, 38.22934335950893]"
3,BEECH ST,473,0458,WILSON AVE,6487,7212,DR WILLIAM G WEATHERS DR,10596,D596,LOCAL,"[-85.80501498815028, 38.228933021550915]","[-85.80483162492965, 38.22753788732054]"
4,GARDEN DR,2442,2470,RAINBOW DR,13394,5391,POPPY WAY,4702,5128,PRIMARY COLLECTOR,"[-85.68020545343364, 38.24806713661984]","[-85.67986300781635, 38.24760815991686]"
5,PARKWAY DR,4573,4974,MOUNT CLAIRE AVE,4162,4476,DEAD END,8594,9811,LOCAL,"[-85.74203979766622, 38.211814770925905]","[-85.74164752771065, 38.21196859682181]"


\# Old code to match centerlines and intersections by SIFID pairs

```py
def build_2(intersection_sifid_index, centerline_sifid_index, 
                      intersection_dict, centerline_dict) -> None:
    # build intersection sifid pair / centerline sifid pair index
    matchset = index_by_SIFID_pairs(intersection_sifid_index, centerline_sifid_index)

    for _, intersection_ids, centerline_ids in matchset.itertuples():
        for int_id in intersection_ids:
            # gather centerline ids into appropriate dictionaries, indexed by intersection ids
            for cl_id in centerline_ids:
                intersection_dict[(int_id, cl_id)] = True
                centerline_dict[(int_id, cl_id)] = True
```

This code was an improved version of this:

```py
def build_match_dicts(intersection_sifid_index, centerline_sifid_index, 
                      intersection_dict, centerline_dict) -> None:
    # build intersection sifid pair / centerline sifid pair index
    matchset = index_by_SIFID_pairs(intersection_sifid_index, centerline_sifid_index)

    for _, intersection_ids, centerline_ids in matchset.itertuples():
        for int_id in intersection_ids:
            # gather centerline ids into appropriate dictionaries, indexed by intersection ids
            intersection_dict[int_id].update(centerline_ids)
            centerline_dict[int_id].update(centerline_ids)
```

Both required error-prone finagling of data objects like this:

```py
# create dictionaries to store future series info
intersections_FST_match = defaultdict(set)
intersections_SEC_match = defaultdict(set)
centerlines_LOW_match = defaultdict(set)
centerlines_HI_match = defaultdict(set)

intm1 = dict()#pd.Series(name='FST_match')
intm2 = dict()#pd.Series(name='SEC_match')
clLOWm = dict()#pd.Series(name='SIFIDLOW_match')
clHIm = dict()#pd.Series(name='SIFIDHI_match')

build_2(intersections_by_FST_SEC, centerlines_by_SIFID_SIFIDLOW, intm1, clLOWm)
build_2(intersections_by_FST_SEC, centerlines_by_SIFID_SIFIDHI, intm1, clHIm)
build_2(intersections_by_SEC_FST, centerlines_by_SIFID_SIFIDLOW, intm2, clLOWm)
build_2(intersections_by_SEC_FST, centerlines_by_SIFID_SIFIDHI, intm2, clHIm)
# build step takes 30s! # not anymore very fast now for some reaason.
```


```python
# Create indexes, sort information into the appropriate dictionaries.
build_match_dicts(intersections_by_FST_SEC, centerlines_by_SIFID_SIFIDLOW, intersections_FST_match, centerlines_LOW_match)
    # Where intersections[FST_SIFID] == centerlines[SIFID] & intersections[SEC_SIFID] == centerlines[SIFIDLOW]
    # Map intersection ids to the centerline ids. Next, do the same for all the other combinations of columns:
build_match_dicts(intersections_by_FST_SEC, centerlines_by_SIFID_SIFIDHI, intersections_FST_match, centerlines_HI_match)
build_match_dicts(intersections_by_SEC_FST, centerlines_by_SIFID_SIFIDLOW, intersections_SEC_match, centerlines_LOW_match)
build_match_dicts(intersections_by_SEC_FST, centerlines_by_SIFID_SIFIDHI, intersections_SEC_match, centerlines_HI_match)

# Convert the dictionaries to Series.
intersections_FST_match = pd.Series(intersections_FST_match, name='FST_match')
intersections_SEC_match = pd.Series(intersections_SEC_match, name='SEC_match')
centerlines_LOW_match = pd.Series(centerlines_LOW_match, name='SIFIDLOW_match')
centerlines_HI_match = pd.Series(centerlines_HI_match, name='SIFIDHI_match')

# Create dataframe by concatenating Series. Intersection id's are the index. 
matchdf = pd.concat((intersections_FST_match, intersections_SEC_match, centerlines_LOW_match, centerlines_HI_match), axis=1)
matchdf.info()
```

```py
intm1 = pd.Series(intm1, name='FST_match', dtype='boolean')
intm2 = pd.Series(intm2, name='SEC_match', dtype='boolean')
clLOWm = pd.Series(clHIm, name='SIFIDLOW_match', dtype='boolean')
clHIm = pd.Series(clLOWm, name='SIFIDHI_match', dtype='boolean')

newmatch = pd.concat((intm1, intm2, clLOWm, clHIm), axis=1)
newmatch[newmatch.SIFIDHI_match & newmatch.SIFIDLOW_match]
```

In [8]:
# get groups
def get_groups(df, by) -> dict:
    return pd.Series(df.groupby(by=by).groups)

centerlines_by_SIFID_SIFIDLOW = get_groups(centerlines, ['SIFID', 'SIFIDLOW'])
centerlines_by_SIFID_SIFIDHI = get_groups(centerlines, ['SIFID', 'SIFIDHI'])

#
#any(centerlines_by_SIFID_SIFIDLOW.apply(len)>1) # True
#any(centerlines_by_SIFID_SIFIDHI.apply(len)>1) # True
#centerlines_by_SIFID_SIFIDHI

intersections_by_FST_SEC = get_groups(intersections, ['FST_SIFID', 'SEC_SIFID'])
# If the index of one of the centerline groups matches an index in this group, that means
#
#   (centerline == centerlines.loc[centerline index])
#   intersection[FST_SIFID] == centerline[SIFID]
#                   and
#   intersections[SEC_SIFID] == either centerline[SIFIDLOW] or centerline[SIFIDHI]
# depending on which set of centerline groups, which we built above, that we are using. 

# The names FST_SIFID / SEC_SIFID -> First / Second intersection SIFID are convention and
# the order is arbitrary. Reversing the order of the index allows us to match SEC_SIFID efficiently.
intersections_by_SEC_FST = get_groups(intersections, ['SEC_SIFID', 'FST_SIFID'])
# If the index of one of the centerline groups is an index in this group, that means
#
#   intersection[SEC_SIFID] == centerline[SIFID]
#                   and
#   intersections[FST_SIFID] == either centerline[SIFIDLOW] or centerline[SIFIDHI]

#
#any(intersections_by_FST_SEC.apply(len) > 1) # True
#any(intersections_by_SEC_FST.apply(len) > 1) # True


def index_by_SIFID_pairs(intersection_sifid_pairs, centerline_sifid_pairs) -> pd.DataFrame:
    return pd.concat((intersection_sifid_pairs, centerline_sifid_pairs), axis=1,
                    # map intersection ids to centerline ids where their sifid pair indexes match
                    # matches are collections of intersection ids/indexes and centerline ids/indexes
                        ).dropna(how='any')
                    # if either collection of ids is empty (or both), there is no match.


In [9]:
mapping = defaultdict(int)

fst_mask = 0b0001
sec_mask = 0b0010
low_mask = 0b0100
hi_mask = 0b1000

for imask, I in ((fst_mask, intersections_by_FST_SEC), (sec_mask, intersections_by_SEC_FST)):
    for cmask, C in ((low_mask, centerlines_by_SIFID_SIFIDLOW), (hi_mask, centerlines_by_SIFID_SIFIDHI)):
        matchset = index_by_SIFID_pairs(I, C)
        set_cols = imask + cmask
        for (s1, s2), intersection_ids, centerline_ids in matchset.itertuples():
            for intersection_id in intersection_ids:
                for cl_id in centerline_ids:
                    mapping[(intersection_id, cl_id)] |= set_cols

mdf = matchdfnumeric = pd.Series(mapping)

matchdfnumeric.index.set_names(('int_id', 'cl_id'), inplace=True)
matchdfnumeric.apply("0b {:04b}".format)

int_id           cl_id 
589991067518338  10250     0b 0101
                 28655     0b 0101
590506463593858  10250     0b 0101
                 28655     0b 0101
409323268213029  26919     0b 0101
                            ...   
847134907955010  174788    0b 1010
847207302035270  175429    0b 1010
847212217628486  175434    0b 1010
847246577432390  175435    0b 1010
847250872399689  175437    0b 1010
Length: 64111, dtype: object

In [10]:
# intersections
iw = matchdfnumeric.sort_index().index.to_frame()
iw

intersection_geos = intersections['GEOMETRY']

int_points = iw.int_id.apply(lambda int_id:intersection_geos.loc[int_id])

# centerlines
cl_GEO_start = centerlines.GEO_start
cl_GEO_end = centerlines.GEO_end

cl_start_points = iw.cl_id.apply(lambda cl_id:cl_GEO_start.loc[cl_id])
cl_end_points = iw.cl_id.apply(lambda cl_id:cl_GEO_end.loc[cl_id])

In [11]:
angle_int_to_end = cl_end_points.combine(int_points, central_angle).sort_index()
angle_int_to_start = cl_start_points.combine(int_points, central_angle).sort_index()


In [12]:
diffs = angle_int_to_start - angle_int_to_end

start_is_close = diffs <= 0
start_is_far = end_is_close = diffs > 0
end_is_far = diffs < 0

def encode(diff):
    if diff < 0: # start angle is smaller
        return 'start' 
    elif diff > 0: # start angle is larger 
        return 'end' 
    elif diff == 0:
        return 'both'
    else:
        return pd.NA 

GEO_map = matchdfnumeric.transform({"int_match": lambda x:x & 0b0011,
                               "hi_match": lambda x:bool(x & 0b1000),
                               "low_match": lambda x:bool(x & 0b0100)})

GEO_map['intersection_point'] = int_points

GEO_map['GEO_match_code'] = diffs.apply(encode)


In [13]:


GEO_map['GEO_close'] = pd.concat((
    cl_start_points[start_is_close],
    cl_end_points[end_is_close]))

GEO_map['GEO_far'] = pd.concat((
    cl_start_points[start_is_far],
    cl_end_points[end_is_far]))

GEO_map['close_angle'] = pd.concat((
    angle_int_to_start[start_is_close],
    angle_int_to_end[end_is_close]))

GEO_map['far_angle'] = pd.concat((
    angle_int_to_start[start_is_far],
    angle_int_to_end[end_is_far]))

GEO_map = GEO_map.sort_index()
GEO_map

int_match  hi_match  low_match  \
int_id           cl_id                                    
5710837346       20353           1     False       True   
                 20860           2     False       True   
                 52803           1      True      False   
10005800273      7109            1      True       True   
                 9233            1     False       True   
...                            ...       ...        ...   
2698443752596326 32774           1     False       True   
                 105004          1     False       True   
2698447711168309 4069            1     False       True   
                 8304            2     False       True   
                 30017           1      True      False   

                                              intersection_point  \
int_id           cl_id                                             
5710837346       20353   [-85.51044384084946, 38.20588660809318]   
                 20860   [-85.51044384084946, 38.20588660809318]   
                 52803   [-85.51044384084946, 38.20588660809318]   
10005800273      7109    [-85.52816882852979, 38.20037561255241]   
                 9233    [-85.52816882852979, 38.20037561255241]   
...                                                          ...   
2698443752596326 32774   [-85.52706694300736, 38.22235647012922]   
                 105004  [-85.52706694300736, 38.22235647012922]   
2698447711168309 4069    [-85.68299954058695, 38.13164510965587]   
                 8304    [-85.68299954058695, 38.13164510965587]   
                 30017   [-85.68299954058695, 38.13164510965587]   

                        GEO_match_code  \
int_id           cl_id                   
5710837346       20353           start   
                 20860           start   
                 52803             end   
10005800273      7109              end   
                 9233            start   
...                                ...   
2698443752596326 32774           start   
                 105004          start   
2698447711168309 4069            start   
                 8304            start   
                 30017             end   

                                                       GEO_close  \
int_id           cl_id                                             
5710837346       20353   [-85.51044384084946, 38.20588660809318]   
                 20860   [-85.51044384084946, 38.20588660809318]   
                 52803   [-85.51044384084946, 38.20588660809318]   
10005800273      7109    [-85.52816882852979, 38.20037561255241]   
                 9233    [-85.52816882852979, 38.20037561255241]   
...                                                          ...   
2698443752596326 32774     [-85.5284541859341, 38.2171605272513]   
                 105004  [-85.52706694300736, 38.22235647012922]   
2698447711168309 4069    [-85.68299954058695, 38.13164510965587]   
                 8304    [-85.68299954058695, 38.13164510965587]   
                 30017   [-85.68299954058695, 38.13164510965587]   

                                                          GEO_far  \
int_id           cl_id                                              
5710837346       20353   [-85.50187141303726, 38.206105324500015]   
                 20860    [-85.50782952965943, 38.20500236063127]   
                 52803    [-85.51425148406123, 38.20488285764657]   
10005800273      7109    [-85.52841872335158, 38.200360349057064]   
                 9233     [-85.52154597701633, 38.20245302533696]   
...                                                           ...   
2698443752596326 32774    [-85.53108855038136, 38.21655402891962]   
                 105004   [-85.52577512279841, 38.21982202058226]   
2698447711168309 4069     [-85.68203371076034, 38.12989574098884]   
                 8304      [-85.6821648141634, 38.13195098098284]   
                 30017    [-85.68417066088023, 38.13377420926451]   

                         close_angle  far_

In [14]:
direct_matches = dm = GEO_map[GEO_map.close_angle == 0]
direct_matches



int_match  hi_match  low_match  \
int_id           cl_id                                    
5710837346       20353           1     False       True   
                 20860           2     False       True   
                 52803           1      True      False   
10005800273      7109            1      True       True   
                 9233            1     False       True   
...                            ...       ...        ...   
2698443752596326 8539            1      True      False   
                 105004          1     False       True   
2698447711168309 4069            1     False       True   
                 8304            2     False       True   
                 30017           1      True      False   

                                              intersection_point  \
int_id           cl_id                                             
5710837346       20353   [-85.51044384084946, 38.20588660809318]   
                 20860   [-85.51044384084946, 38.20588660809318]   
                 52803   [-85.51044384084946, 38.20588660809318]   
10005800273      7109    [-85.52816882852979, 38.20037561255241]   
                 9233    [-85.52816882852979, 38.20037561255241]   
...                                                          ...   
2698443752596326 8539    [-85.52706694300736, 38.22235647012922]   
                 105004  [-85.52706694300736, 38.22235647012922]   
2698447711168309 4069    [-85.68299954058695, 38.13164510965587]   
                 8304    [-85.68299954058695, 38.13164510965587]   
                 30017   [-85.68299954058695, 38.13164510965587]   

                        GEO_match_code  \
int_id           cl_id                   
5710837346       20353           start   
                 20860           start   
                 52803             end   
10005800273      7109              end   
                 9233            start   
...                                ...   
2698443752596326 8539              end   
                 105004          start   
2698447711168309 4069            start   
                 8304            start   
                 30017             end   

                                                       GEO_close  \
int_id           cl_id                                             
5710837346       20353   [-85.51044384084946, 38.20588660809318]   
                 20860   [-85.51044384084946, 38.20588660809318]   
                 52803   [-85.51044384084946, 38.20588660809318]   
10005800273      7109    [-85.52816882852979, 38.20037561255241]   
                 9233    [-85.52816882852979, 38.20037561255241]   
...                                                          ...   
2698443752596326 8539    [-85.52706694300736, 38.22235647012922]   
                 105004  [-85.52706694300736, 38.22235647012922]   
2698447711168309 4069    [-85.68299954058695, 38.13164510965587]   
                 8304    [-85.68299954058695, 38.13164510965587]   
                 30017   [-85.68299954058695, 38.13164510965587]   

                                                          GEO_far  \
int_id           cl_id                                              
5710837346       20353   [-85.50187141303726, 38.206105324500015]   
                 20860    [-85.50782952965943, 38.20500236063127]   
                 52803    [-85.51425148406123, 38.20488285764657]   
10005800273      7109    [-85.52841872335158, 38.200360349057064]   
                 9233     [-85.52154597701633, 38.20245302533696]   
...                                                           ...   
2698443752596326 8539      [-85.52800654883289, 38.2245264344245]   
                 105004   [-85.52577512279841, 38.21982202058226]   
2698447711168309 4069     [-85.68203371076034, 38.12989574098884]   
                 8304      [-85.6821648141634, 38.13195098098284]   
                 30017    [-85.68417066088023, 38.13377420926451]   

                         close_angle  far_

In [18]:

int_FST_cl_match = defaultdict(list)
int_SEC_cl_match = defaultdict(list)

cl_low_int_match = defaultdict(list)
cl_hi_int_match = defaultdict(list)

cl_start_int_match = defaultdict(list)
cl_end_int_match = defaultdict(list)

hi_xor_low = dm[dm.hi_match ^ dm.low_match]
both = dm[dm.hi_match & dm.low_match]
#dm[~dm.hi_match & ~dm.low_match] # None

#both[both.GEO_match_code == 'both']


In [279]:
# HERE

def order(sifid1, sifid2):
    if sifid1 > sifid2:
        return (sifid2, sifid1)
    else:
        return (sifid1, sifid2)

int_SIFID_order = intersections.FST_SIFID.combine(intersections.SEC_SIFID, order)
intersections_by_SIFID_order = pd.Series(intersections.groupby(int_SIFID_order).groups)

cl_SIFID_LOW_order = centerlines.SIFID.combine(centerlines.SIFIDLOW, order)
centerlines_by_SIFID_LOW_order = pd.Series(centerlines.groupby(cl_SIFID_LOW_order).groups)

cl_SIFID_HI_order = centerlines.SIFID.combine(centerlines.SIFIDHI, order)
centerlines_by_SIFID_HI_order = pd.Series(centerlines.groupby(cl_SIFID_HI_order).groups)


#cl_SIFID_LOW_order.combine(centerlines_by_SIFID_LOW_order, lambda x,y:(x,y))
centerlines_by_SIFID_order = centerlines_by_SIFID_LOW_order.combine(centerlines_by_SIFID_HI_order,
                           lambda x, y:x.union(y),
                           fill_value = pd.Index(()))


In [330]:
ic_map = pd.concat((
intersections_by_SIFID_order,
centerlines_by_SIFID_order), axis=1,
keys=['intersection_ids', 'centerline_ids'])

ic_map = ic_map.dropna() # ignore mismatches for now

int_cl_SIFID_map = defaultdict(set)

def map_int_to_cl(row):
    int_ids = row.intersection_ids
    cl_ids = row.centerline_ids
    for int_id in int_ids:
        int_cl_SIFID_map[int_id].update(cl_ids)

ic_map.apply(map_int_to_cl, axis=1)
int_cl_SIFID_map = pd.Series(int_cl_SIFID_map).sort_index()
int_cl_SIFID_map


5710837346                      {20353, 52803, 20860}
10005800273         {13504, 7109, 84838, 31468, 9233}
14300767569         {13504, 7109, 84838, 31468, 9233}
79566575913                     {26604, 90022, 23911}
83861558290                             {26604, 4511}
                                  ...                
2698419891660598                {19602, 12019, 19268}
2698428751996728                  {26925, 6101, 1638}
2698437341931319                 {5496, 19602, 26925}
2698443752596326    {8034, 32774, 6763, 105004, 8539}
2698447711168309                  {8304, 30017, 4069}
Length: 19738, dtype: object

In [336]:
cl_gg_start = centerlines.groupby(centerlines.GEO_start.apply(tuple)).groups 
# annoying converting between types
cl_gg_end = centerlines.groupby(centerlines.GEO_end.apply(tuple)).groups

empty = set()

def match_cl_point(point, which=None):
    this = which.get(point, None)
    if this is None:
        return empty
    else:
        return this
        
int_GEO = intersections.GEOMETRY.apply(tuple)
start_match = int_GEO.apply(match_cl_point, which=cl_gg_start)
end_match = int_GEO.apply(match_cl_point, which=cl_gg_end)


In [353]:

gsm = int_cl_SIFID_map.combine(start_match, lambda x, y:x.intersection(y), fill_value=set())
#start_match
c = 0
for row in gsm.items():
    int_id, cl_ids = row
    os = start_match.loc[int_id]
    if len(cl_ids) != len(os):
        c += 1
        print(int_id, cl_ids, os)

c

87315224873 set() Index([90020], dtype='int64', name='OBJECTID')
1404457620489 set() Index([8261, 25468], dtype='int64', name='OBJECTID')
1602451677192 {30306} Index([3131, 9407, 30306], dtype='int64', name='OBJECTID')
1602451685766 {9407} Index([3131, 9407, 30306], dtype='int64', name='OBJECTID')
1602560213382 {3131} Index([3131, 9407, 30306], dtype='int64', name='OBJECTID')
2410644330103 set() Index([8189, 31364], dtype='int64', name='OBJECTID')
2410644334353 {31364} Index([8189, 31364], dtype='int64', name='OBJECTID')
2410927318801 {8189} Index([8189, 31364], dtype='int64', name='OBJECTID')
2418358114624 {50572, 18519} Index([10901, 18519, 50572], dtype='int64', name='OBJECTID')
2418358146905 set() Index([10901, 18519, 50572], dtype='int64', name='OBJECTID')
2419564009305 {10901, 18519} Index([10901, 18519, 50572], dtype='int64', name='OBJECTID')
2488403256657 {3457} Index([3457, 6763, 42246], dtype='int64', name='OBJECTID')
2762860829078 {497} Index([497, 160071], dtype='int64', na

2625

In [347]:
start_match.sort_index()

INTID
5710837346          Index([20353, 20860], dtype='int64', name='OBJ...
10005800273         Index([9233, 13504], dtype='int64', name='OBJE...
14300767569             Index([7109], dtype='int64', name='OBJECTID')
18011691414                                                        {}
23945910678                                                        {}
                                          ...                        
2698419891660598    Index([19268, 19602], dtype='int64', name='OBJ...
2698428751996728    Index([1638, 6101], dtype='int64', name='OBJEC...
2698437341931319    Index([5496, 26925], dtype='int64', name='OBJE...
2698443752596326      Index([105004], dtype='int64', name='OBJECTID')
2698447711168309    Index([4069, 8304], dtype='int64', name='OBJEC...
Name: GEOMETRY, Length: 20946, dtype: object

In [ ]:
wm = both[both.GEO_match_code != 'both']
wm

def gm(row):
    int_id, cl_id = row.name

    cl_geo_close = row.GEO_close
    cl_geo_far = row.GEO_far 
    int_point = row.intersection_point 

    cl = centerlines.loc[cl_id]
    cl_SIFID = cl.SIFID 
    low_cross_sifid = cl.SIFIDLOW
    hi_cross_sifid = cl.SIFIDHI

    try:
        centerlines_by_SIFID_SIFIDLOW[low_cross_sifid, cl_SIFID]
    except KeyError:
        pass
    try:
        centerlines_by_SIFID_SIFIDHI[hi_cross_sifid, cl_SIFID]
    except KeyError:
        pass

        
    cl_low_geo_start = ...
    cl_low_geo_end = ...

    cl_hi_geo_start = ...
    cl_hi_geo_end = ...


int_match  hi_match  low_match  \
int_id           cl_id                                   
10005800273      7109           1      True       True   
14300767569      7109           1      True       True   
91610177636      31054          2      True       True   
582312564550     7992           1      True       True   
                 21943          2      True       True   
...                           ...       ...        ...   
2697659918312745 4934           1      True       True   
2698219353024632 27847          1      True       True   
2698235477223204 96             1      True       True   
                 6074           2      True       True   
2698443752596326 8034           2      True       True   

                                              intersection_point  \
int_id           cl_id                                             
10005800273      7109    [-85.52816882852979, 38.20037561255241]   
14300767569      7109   [-85.52841872335158, 38.200360349057064]   
91610177636      31054  [-85.49041133329159, 38.204811157321686]   
582312564550     7992    [-85.48918423050128, 38.25469997424565]   
                 21943   [-85.48918423050128, 38.25469997424565]   
...                                                          ...   
2697659918312745 4934   [-85.59552658739156, 38.118673033102404]   
2698219353024632 27847   [-85.55849963244644, 38.24249765745603]   
2698235477223204 96      [-85.60771003631034, 38.32844290109257]   
                 6074    [-85.60771003631034, 38.32844290109257]   
2698443752596326 8034    [-85.52706694300736, 38.22235647012922]   

                       GEO_match_code  \
int_id           cl_id                  
10005800273      7109             end   
14300767569      7109           start   
91610177636      31054            end   
582312564550     7992             end   
                 21943            end   
...                               ...   
2697659918312745 4934             end   
2698219353024632 27847            end   
2698235477223204 96             start   
                 6074           start   
2698443752596326 8034             end   

                                                       GEO_close  \
int_id           cl_id                                             
10005800273      7109    [-85.52816882852979, 38.20037561255241]   
14300767569      7109   [-85.52841872335158, 38.200360349057064]   
91610177636      31054  [-85.49041133329159, 38.204811157321686]   
582312564550     7992    [-85.48918423050128, 38.25469997424565]   
                 21943   [-85.48918423050128, 38.25469997424565]   
...                                                          ...   
2697659918312745 4934   [-85.59552658739156, 38.118673033102404]   
2698219353024632 27847   [-85.55849963244644, 38.24249765745603]   
2698235477223204 96      [-85.60771003631034, 38.32844290109257]   
                 6074    [-85.60771003631034, 38.32844290109257]   
2698443752596326 8034    [-85.52706694300736, 38.22235647012922]   

                                                         GEO_far  close_angle  \
int_id           cl_id                                                          
10005800273      7109   [-85.52841872335158, 38.200360349057064]          0.0   
14300767569      7109    [-85.52816882852979, 38.20037561255241]          0.0   
91610177636      31054   [-85.49945278036108, 38.20743661819182]          0.0   
582312564550     7992    [-85.49091110522652, 38.25286219846074]          0.0   
                 21943   [-85.49091110522652, 38.25286219846074]          0.0   
...                                                          ...          ...   
2697659918312745 4934    [-85.59973607672718, 38.12006181925096]          0.0   
2698219353024632 27847   [-85.55812792258003, 38.24290938778276]          0.0   
2698235477223204 96     [-85.60544072029754, 38.329357281872326]          0.0   
                 6074   [-85.60544072029754, 38.329357281872326]          

In [293]:
#cl_gg_start = centerlines.groupby(centerlines.GEO_start.apply(tuple)).groups
#cl_gg_end = centerlines.groupby(centerlines.GEO_end.apply(tuple)).groups

int_gg = intersections.groupby(intersections.GEOMETRY.apply(tuple)).groups

#gg_map = pd.concat((int_gg, cl_gg_start, cl_gg_end), axis=1, keys=['int_ids', 'cl_start', 'cl_end'])


In [370]:


#me = defaultdict(int)
#ms = defaultdict(int) 
mb = defaultdict(int)

for int_point, int_ids in int_gg.items():

    starts = cl_gg_start.get(int_point, None)
    if starts is not None:
        for int_id in int_ids:
            for cl_id in starts:
                mb[(int_id, cl_id)] |= 0b10

    ends = cl_gg_end.get(int_point, None)
    if ends is not None:
        for int_id in int_ids:
            for cl_id in ends:
                mb[(int_id, cl_id)] |= 0b01

mb = pd.Series(mb, name='point_matches')
mb.index.set_names(['int_id', 'cl_id'], inplace=True)
mb

int_id           cl_id 
607025278104711  1334      2
                 4613      2
                 11316     1
607020983137801  11316     2
                 20599     1
                          ..
709691547185296  38147     2
                 38158     1
726338898139543  108844    1
726784057799682  108844    2
                 177349    1
Name: point_matches, Length: 66474, dtype: int64

In [372]:
hh = pd.concat((mb, matchdfnumeric), axis=1).convert_dtypes()
hh.dropna()

point_matches   0
int_id          cl_id                   
607025278104711 1334               2   6
                4613               2   5
                11316              1   9
607020983137801 11316              2   5
                20599              1  10
...                              ...  ..
709671528034448 38147              1  10
709684886827152 38123              1  10
                38130              1   9
709691547185296 38147              2   6
                38158              1   9

[58707 rows x 2 columns]

In [235]:
ii=hh[hh[0].notna() & hh[1].notna()]
ii[ii[0]==3]

,,0,1
int_id,cl_id,,
447953572487975,24013,3,13
422433206461753,27636,3,14
2696517125592441,7478,3,14
690789222245942,6389,3,14


In [ ]:

matches = gg_map.int_ids.notna() & (gg_map.cl_start.notna() | gg_map.cl_end.notna())
gg_mm = gg_map[matches]


,,int_ids,cl_start,cl_end
-85.936859,38.005901,"Index([607025278104711], dtype='int64', name='...","Index([1334, 4613], dtype='int64', name='OBJEC...","Index([11316], dtype='int64', name='OBJECTID')"
-85.935704,38.006712,"Index([607020983137801], dtype='int64', name='...","Index([11316], dtype='int64', name='OBJECTID')","Index([20599, 21796], dtype='int64', name='OBJ..."
-85.926923,38.012872,"Index([607007728145938], dtype='int64', name='...","Index([2808, 21796], dtype='int64', name='OBJE...","Index([15032], dtype='int64', name='OBJECTID')"
-85.921471,38.018520,"Index([623530856224919], dtype='int64', name='...","Index([22217], dtype='int64', name='OBJECTID')","Index([22427, 25011], dtype='int64', name='OBJ..."
-85.921173,38.018295,"Index([623565197154097], dtype='int64', name='...","Index([15032, 25011], dtype='int64', name='OBJ...","Index([27330], dtype='int64', name='OBJECTID')"
...,...,...,...,...
-85.406573,38.125504,"Index([709671528034448], dtype='int64', name='...","Index([38122, 38123], dtype='int64', name='OBJ...","Index([38147], dtype='int64', name='OBJECTID')"
-85.404792,38.127515,"Index([709684886827152], dtype='int64', name='...",0,"Index([38123, 38130], dtype='int64', name='OBJ..."
-85.399015,38.115176,"Index([709691547185296], dtype='int64', name='...","Index([38147], dtype='int64', name='OBJECTID')","Index([38158], dtype='int64', name='OBJECTID')"
-85.347920,38.206860,"Index([726338898139543], dtype='int64', name='...",0,"Index([108844], dtype='int64', name='OBJECTID')"


In [ ]:

def match_ids(gf):
    gf = gf.fillna('')
    se_map = defaultdict(lambda :[False, False])
    for row in gf.itertuples():
        int_ids = row.int_ids
        starts = row.cl_start
        ends = row.cl_end 
        for int_id in int_ids:
            for cl_start in starts:
                se_map[(int_id, cl_start)][0] = True 
            for cl_end in ends:
                se_map[(int_id, cl_end)][1] = True 
    return se_map
    

se = match_ids(gg_mm)
#gg_mm.fillna('')
pd.DataFrame.from_dict



ValueError: Expected 'index', 'columns' or 'tight' for orient parameter. Got 'records' instead

In [ ]:


c = centerlines_by_SIFID_SIFIDHI.combine(
    centerlines_by_SIFID_SIFIDLOW, lambda x, y:x.union(y), fill_value=pd.Index(()))

def get_cl_cross_geos(cl_id):
    cl = centerlines.loc[cl_id]
    sifid = cl.SIFID 
    low = cl.SIFIDLOW
    hi = cl.SIFIDHI 

    try:
        low_cl_match  = c.loc[low, sifid]
    except KeyError:
        low_cl_match = None

    try:
        hi_cl_match = c.loc[hi, sifid]
    except KeyError:
        if low_cl_match is None:
            return "no match"
    
    


1      591      Index([6313, 10250, 15159, 28655], dtype='int6...
       897                         Index([26919], dtype='object')
       906      Index([29463, 31519], dtype='int64', name='OBJ...
       1813                        Index([11063], dtype='object')
       2252                        Index([25407], dtype='object')
                                      ...                        
15640  8594                       Index([180227], dtype='object')
15641  2072                       Index([180229], dtype='object')
       8594                       Index([180229], dtype='object')
15642  8594                       Index([180236], dtype='object')
       15611                      Index([180236], dtype='object')
Length: 43418, dtype: object

In [ ]:

def do(row):
    sif = row.SIFID 
    low = row.SIFIDLOW
    hi = row.SIFIDHI
    geo_start = row.GEO_start
    geo_end = row.GEO_end 

    try:
        low_match = c.loc[low, sif]
    except KeyError:
        low_match = None

    try:
        hi_match = c.loc[hi, sif]
    except KeyError:
        hi_match = None 
    
    return {"low_match":low_match, "hi_match":hi_match}

centerlines.apply(do, axis=1)



OBJECTID
1                   {'low_match': [6013], 'hi_match': None}
2         {'low_match': [3753, 23064], 'hi_match': [2808...
3         {'low_match': [20485, 24494, 27237], 'hi_match...
4         {'low_match': [84286, 84288], 'hi_match': [842...
5                  {'low_match': [18357], 'hi_match': None}
                                ...                        
180232            {'low_match': None, 'hi_match': [145324]}
180233        {'low_match': [180235], 'hi_match': [180236]}
180234        {'low_match': [180236], 'hi_match': [180235]}
180235    {'low_match': [176706, 176711, 180233, 180234]...
180236    {'low_match': [180233, 180234], 'hi_match': None}
Length: 34215, dtype: object

In [ ]:
def cl_geo_match(row):
    int_id, cl_id = row.name
    intersection = intersections.loc[int_id]
    centerline = centerlines.loc[cl_id]


In [ ]:

def do(row):
    int_id, cl_id = row.name

    int_match = row.int_match
    assert int_match != 0b11

    low_match = row.low_match
    hi_match = row.hi_match
    GEO_match = row.GEO_match_code

    if low_match ^ hi_match:
        if GEO_match != 'both':
            # normal case
            if int_match == 1:
                int_FST_cl_match[int_id].append(cl_id)
            else: # int_match == 2:
                int_SEC_cl_match[int_id].append(cl_id)
            
            if GEO_match == 'hi':
                cl_hi_int_match[cl_id].append(int_id) 
            else: # GEO_match == 'low' 
                cl_low_int_match[cl_id].append(int_id)
        else:
            # impossible for this data
            raise ValueError
    elif low_match & hi_match:
        if GEO_match == 'both':
            # easy?
            if int_match == 1:
                int_FST_cl_match[int_id].append(cl_id)
            elif  int_match == 2: 
                int_SEC_cl_match[int_id].append(cl_id)
            cl_hi_int_match[cl_id].append(int_id) 
            cl_low_int_match[cl_id].append(int_id)
        else:
            # most complex case
            cl_geo_match(row)
            ...
    else:
        # impossible case
        raise ValueError
    
ee = direct_matches.apply(do, axis=1)

In [115]:
len(both[both.GEO_match_code != 'both']) == len(not_m) # True. Good news
not_m

[(10005800273, 7109),
 (14300767569, 7109),
 (91610177636, 31054),
 (582312564550, 7992),
 (582312564550, 21943),
 (1119183476550, 7992),
 (1119183476550, 21943),
 (1379545420614, 25745),
 (1413953435207, 4267),
 (1444018206279, 4267),
 (1478966155554, 14679),
 (1478966155554, 31731),
 (1483261122850, 14679),
 (1483261122850, 31731),
 (1521279341382, 25745),
 (1650764847510, 4286),
 (1650764847510, 7578),
 (2204250564471, 29651),
 (2208545531767, 29651),
 (2574101274724, 31054),
 (2762860829078, 931),
 (3510485686678, 7578),
 (3719459635475, 17799),
 (3719459635475, 29332),
 (3719459635475, 30005),
 (3801064014099, 29332),
 (3801064014099, 31236),
 (4702205957409, 4158),
 (4706500924705, 4158),
 (4710859920674, 16624),
 (4710859920674, 19924),
 (4715154887970, 16624),
 (4715154887970, 19924),
 (4849707002201, 20792),
 (4879771773273, 20792),
 (5923683146134, 24566),
 (6017656526883, 25208),
 (8127274981782, 18901),
 (8131569949078, 8772),
 (8131569949078, 18901),
 (8140159883670, 931),

In [110]:
len(not_m)

2023

In [ ]:
def do(row):
    sif = row.SIFID

    low = row.SIFIDLOW
    hi = row.SIFIDHI 

    low_pair = (low, sif)
    hi_pair = (hi, sif)


    for memo, cross in ((set(), low), (set(), hi)):
        pair = (cross, sif)
        for table in (centerlines_by_SIFID_SIFIDHI, centerlines_by_SIFID_SIFIDLOW):
            try:
                match = table.loc[pair]
            except KeyError:
                continue
            else:
                memo = match.union(memo)



,ROADNAME,SIFID,SIFCODE,low_cross_ROADNAME,SIFIDLOW,LOCROSSSIF,hi_cross_ROADNAME,SIFIDHI,HICROSSSIF,CORE_CLASS,GEO_start,GEO_end
OBJECTID,,,,,,,,,,,,
195,GOLDSMITH LN,2543,2585,DAVID GRAVES DR,14199,E998,BELMONT RD,495,0483,PRIMARY COLLECTOR,"[-85.67457458991743, 38.20539083374192]","[-85.67334993258598, 38.20602795942558]"
198,GOLDSMITH LN,2543,2585,DAVID GRAVES DR,14199,E998,SUMNER RD,5610,6210,PRIMARY COLLECTOR,"[-85.67201567611251, 38.206724595474846]","[-85.67064558883953, 38.20743107542566]"


In [ ]:

c = centerlines_by_SIFID_SIFIDHI.combine(
    centerlines_by_SIFID_SIFIDLOW, lambda x, y:x.union(y), fill_value=pd.Index(()))




In [ ]:

def match(cl):
    sifid = cl.SIFID
    low = cl.SIFIDLOW
    hi = cl.SIFIDHI

    try:
        low_match = c.loc[low, sifid]
    except KeyError:
        low_match = None
    try:
        hi_match = c.loc[hi, sifid]
    except KeyError:
        hi_match = None 

    return {"low_match":low_match, "hi_match": hi_match}


centerlines.apply(match, axis=1)

#centerlines.head(150).tail()
#centerlines.loc[centerlines_by_SIFID_SIFIDLOW.loc[2543, 14199]]

In [ ]:

def do_geo(row):
    int_id, cl_id = row.name
    if row.int_match == 1:
        int_FST_cl_match[int_id].append(cl_id)
    elif row.int_match == 2:
        int_SEC_cl_match[int_id].append(cl_id)

    geo_match = row.GEO_match_code
    if geo_match == 'start':
        cl_start_int_match[cl_id].append(int_id)
    elif geo_match == 'end':
        cl_end_int_match[cl_id].append(int_id)
    elif geo_match == 'both':
        cl_start_int_match[cl_id].append(int_id)
        cl_end_int_match[cl_id].append(int_id)

direct_matches.apply(do_geo, axis=1)

geos = pd.DataFrame.from_dict({'FST_match':int_FST_cl_match, 'SEC_match':int_SEC_cl_match})
(geos.FST_match.dropna().apply(len) > 3).any()

In [ ]:


mm = GEO_map[['int_match', 'hi_match', 'low_match', 'GEO_match_code',
               'close_angle', 'intersection_point', 'GEO_close']]
dm = mm[mm.close_angle == 0]
dm




In [729]:
dm[(dm.hi_match == True) & (dm.low_match == True)]
#) & (dm.GEO_match_code != 'both')]

int_match  hi_match  low_match GEO_match_code  \
int_id           cl_id                                                  
10005800273      7109           1      True       True            end   
14300767569      7109           1      True       True          start   
91610177636      31054          2      True       True            end   
582312564550     7992           1      True       True            end   
                 21943          2      True       True            end   
...                           ...       ...        ...            ...   
2697659918312745 4934           1      True       True            end   
2698219353024632 27847          1      True       True            end   
2698235477223204 96             1      True       True          start   
                 6074           2      True       True          start   
2698443752596326 8034           2      True       True            end   

                        close_angle                        intersection_point  \
int_id           cl_id                                                          
10005800273      7109           0.0   [-85.52816882852979, 38.20037561255241]   
14300767569      7109           0.0  [-85.52841872335158, 38.200360349057064]   
91610177636      31054          0.0  [-85.49041133329159, 38.204811157321686]   
582312564550     7992           0.0   [-85.48918423050128, 38.25469997424565]   
                 21943          0.0   [-85.48918423050128, 38.25469997424565]   
...                             ...                                       ...   
2697659918312745 4934           0.0  [-85.59552658739156, 38.118673033102404]   
2698219353024632 27847          0.0   [-85.55849963244644, 38.24249765745603]   
2698235477223204 96             0.0   [-85.60771003631034, 38.32844290109257]   
                 6074           0.0   [-85.60771003631034, 38.32844290109257]   
2698443752596326 8034           0.0   [-85.52706694300736, 38.22235647012922]   

                                                       GEO_close  
int_id           cl_id                                            
10005800273      7109    [-85.52816882852979, 38.20037561255241]  
14300767569      7109   [-85.52841872335158, 38.200360349057064]  
91610177636      31054  [-85.49041133329159, 38.204811157321686]  
582312564550     7992    [-85.48918423050128, 38.25469997424565]  
                 21943   [-85.48918423050128, 38.25469997424565]  
...                                                          ...  
2697659918312745 4934   [-85.59552658739156, 38.118673033102404]  
2698219353024632 27847   [-85.55849963244644, 38.24249765745603]  
2698235477223204 96      [-85.60771003631034, 38.32844290109257]  
                 6074    [-85.60771003631034, 38.32844290109257]  
2698443752596326 8034    [-85.52706694300736, 38.22235647012922]  

[2027 rows x 7 columns]

In [ ]:
int_FST_cl = defaultdict(list)
int_SEC_cl = defaultdict(list)

cl_LOW_int = defaultdict(list)
cl_HI_int = defaultdict(list)
cl_start_int = defaultdict(list)
cl_end_int = defaultdict(list)


def do(row):
    int_id, cl_id = row.name
    intersection = intersections.loc[int_id]
    centerline = centerlines.loc[cl_id]

    cl_SIFID = centerline.SIFID
    int_FST = intersection.FST_SIFID
    int_SEC = intersection.SEC_SIFID
    if row.int_match == 1:
        assert cl_SIFID == int_FST
        int_FST_cl[int_id].append(cl_id)
        int_cross_SIFID = int_SEC
    elif row.int_match == 2:
        assert cl_SIFID == int_SEC
        int_SEC_cl[int_id].append(cl_id)
        int_cross_SIFID = int_FST
    
    if row.cl_match == 'low':
        assert centerline.SIFIDLOW == int_cross_SIFID
        cl_LOW_int[cl_id].append(int_id)
    elif row.cl_match == 'hi':
        assert centerline.SIFIDHI == int_cross_SIFID
        cl_HI_int[cl_id].append(int_id)

    if row.GEO_match_code == 'start':
        cl_start_int[cl_id].append(int_id)
    elif row.GEO_match_code == 'end':
        cl_end_int[cl_id].append(int_id)



#point_match = mm[mm.close_angle == 0]
dm.apply(do, axis=1)
# how to deal with 'boths' ??


int_m = pd.DataFrame.from_dict({'int_FST_cl': int_FST_cl,
                        'int_SEC_cl': int_SEC_cl})

cl_m = pd.DataFrame.from_dict({
    'cl_LOW_int': cl_LOW_int,
    'cl_HI_int': cl_HI_int,
    'cl_start_int': cl_start_int,
    'cl_end_int': cl_end_int,})


In [642]:
cf = cl_m.cl_start_int.dropna()
cf[cf.apply(len) == 2]


18519         [2418358114624, 2419564009305]
31991       [20160850481987, 20160850483577]
29458       [21376879845493, 21376879849536]
30767       [39583384491283, 39583845937458]
27033      [43316244673058, 712767631022432]
                         ...                
28620     [728388906197030, 728393796624422]
132206    [728388906197030, 728398091600387]
151719    [730706905155414, 730736441292164]
170641    [846043974596247, 846168528713367]
170633    [846086924334743, 846629611828887]
Name: cl_start_int, Length: 78, dtype: object

In [662]:
def get_ints(*ints):
    for int in ints:
        display(intersections.loc[int])

get_ints(10005800273)

FST_ROADNAME                                    REHL RD
SEC_ROADNAME                          TUCKER STATION RD
SIFCODE1                                           5464
SIFCODE2                                           6551
FST_SIFID                                          4976
SEC_SIFID                                          5908
GEOMETRY        [-85.52816882852979, 38.20037561255241]
Name: 10005800273, dtype: object

In [661]:
def get_cls(*cls):
    for cl in cls:
        display(centerlines.loc[cl])

get_cls(	7109)

ROADNAME                                               REHL RD
SIFID                                                     4976
SIFCODE                                                   5464
low_cross_ROADNAME                           TUCKER STATION RD
SIFIDLOW                                                  5908
LOCROSSSIF                                                6551
hi_cross_ROADNAME                            TUCKER STATION RD
SIFIDHI                                                   5908
HICROSSSIF                                                6551
CORE_CLASS                                               LOCAL
GEO_start             [-85.52841872335158, 38.200360349057064]
GEO_end                [-85.52816882852979, 38.20037561255241]
Name: 7109, dtype: object

In [ ]:
def lem(xs):
    if pd.isna(xs):
        return 0
    else:
        return len(xs)

cc = cl_m.cl_LOW_int.dropna().apply(len)

def j(cls):
    ...
    


int_m.int_FST_cl.dropna().apply(j).all()

np.True_

In [ ]:
S = pd.Series


int_FST_cl = S(int_FST_cl)
int_SEC_cl = S(int_SEC_cl)

cl_LOW_int = S(cl_LOW_int)
cl_HI_int = S(cl_HI_int)
#S(cl_start_int).apply(lambda x:len(x) == 1).all() # False
#S(cl_end_int).apply(lambda x:len(x) == 1).all() # False

cl_HI_int[cl_HI_int.apply(len) >1]


5710837346                 [20860]
10005800273                [13504]
14300767569                [84838]
79566575913         [23911, 90022]
83861558290                 [4511]
                         ...      
2698381254387508       [576, 9388]
2698419891660598           [19602]
2698428751996728            [6101]
2698437341931319            [5496]
2698447711168309            [8304]
Length: 18488, dtype: object

In [549]:

t = cl_LOW_int[cl_LOW_int.apply(len) == 2] # some cls have 2 intersection matches
# none have more than that
# more than one intersection matches on the cl
# this shouldn't be the case, logically. # clarify?

display(t) # 66 rows


67216       [7095824117768, 721281507401747]
25217       [20303981091221, 20355520698773]
27672       [22235165530518, 92591024806294]
598         [41795557880148, 41804147814740]
119721      [41957824364690, 42198342533266]
                         ...                
174155    [844656661558820, 847057579734084]
162317    [844738298200420, 844739559880241]
162640    [844802722952754, 844811312887346]
167433    [845603724072841, 845612314007433]
163913    [847182152726083, 847186447693379]
Length: 66, dtype: object

In [ ]:


ixs = list()

for cl, ints in t.items():
    for i in ints:
        ixs.append((i, cl))

dm.loc[ixs]


In [ ]:

def test1(point):
    a, b = point
    if a != 0:
        return False
    if b != 0:
        return False
    return True 

(imm.intersection_point - imm.GEO_close).apply(test1)

In [ ]:
""" 
columns to add

centerlines -> 
    low_geo, hi_geo == mark start/end of geometry OR align geometry order? 
    low_int, hi_int = intersections

    find
    next_hi -> connection through intersection
    next_low -> connection though other intersection

intersections ->
    fst_roadway == ordered pair
    sec_roadway == ordered pair

    find 
    next intersections -> follow through each roadway component
"""


In [296]:
# -> create# NAME_df


# #int_d = intersections[['FST_ROADNAME', 'SEC_ROADNAME']]
# cl_d = centerlines[['ROADNAME', 'SIFIDLOW', 'SIFIDHI']]

# def get_int_names(int_id):
#     int_d.loc[int_id]

# def get_cl_names(cl_id):
#     rw, sl, sh = cl_d.loc[cl_id]
#     sl = sifid_to_roadname.get(sl)
#     sh = sifid_to_roadname.get(sh)
#     return pd.Series({
#         'ROADNAME': rw,
#         'LO_cross': sl,
#         'HI_cross': sh})

# NAME_df = iw.cl_id.apply(lambda id:centerlines.loc[id])
# #matchdfnumeric.index.to_frame()
# #4.9 sec

# NAME_df

matchdfnumeric[matchdfnumeric.apply(lambda x:((x >> 2) == 3))]

int_id           cl_id 
323827650687044  18779     13
323879190294596  18779     13
40699115667584   13360     13
40922453966976   13360     13
319438198302071  22602     13
                           ..
847177857758787  174793    14
845631654655585  167436    14
845627359688289  167436    14
847130612987714  174475    14
847134907955010  174475    14
Length: 2521, dtype: int64

In [ ]:
centerlines

low_group = ['low_cross_ROADNAME', 'SIFIDLOW', 'LOCROSSSIF']
hi_group = ['hi_cross_ROADNAME', 'SIFIDHI', 'HICROSSSIF']

cols = ['low_cross_ROADNAME', 'SIFIDLOW', 'LOCROSSSIF', 'hi_cross_ROADNAME', 'SIFIDHI', 'HICROSSSIF']

mm= pd.concat((matchdfnumeric.apply(lambda x:x>>2), GEO_map.GEO_match_code), axis=1)

GEO_map[['intersection_point', 'GEO_match_code', 'GEO_close']].assign(mdf=matchdfnumeric)

matchdfnumeric.transform({"int_match": lambda x:x & 0b0011,
                          "LOW_match": lambda x:bool(x & low_mask),
                          "HI_match": lambda x:bool(x & hi_mask)})


int_match  LOW_match  HI_match
int_id          cl_id                                 
589991067518338 10250           1       True     False
                28655           1       True     False
590506463593858 10250           1       True     False
                28655           1       True     False
409323268213029 26919           1       True     False
...                           ...        ...       ...
847134907955010 174788          2      False      True
847207302035270 175429          2      False      True
847212217628486 175434          2      False      True
847246577432390 175435          2      False      True
847250872399689 175437          2      False      True

[64111 rows x 3 columns]

In [314]:

#SIF_columns = ["SIFIDLOW", "SIFIDHI", "HICROSSSIF", "LOCROSSSIF"]
SIF_columns = ["SIFID", "SIFIDLOW", "SIFIDHI"]

SIF_data = centerlines[SIF_columns]

ee = iw.cl_id.apply(lambda cl_id: SIF_data.loc[cl_id])#.assign(int_match=matchdfnumeric & 0b0011)
SIF_data
ee

SIFID  SIFIDLOW  SIFIDHI
int_id           cl_id                           
5710837346       20353    4976      6856     6857
                 20860    6856      4976     8594
                 52803    4976      4674     6856
10005800273      7109     4976      5908     5908
                 9233     4976      5908    12697
...                        ...       ...      ...
2698443752596326 32774    5908       351     1117
                 105004   5908       351    15012
2698447711168309 4069     4716     12859     5897
                 8304    12859      4716     8594
                 30017    4716       586    12859

[64111 rows x 3 columns]

SIFID  SIFIDLOW  SIFIDHI
int_id           cl_id                           
5710837346       20353    4976      6856     6857
                 20860    6856      4976     8594
                 52803    4976      4674     6856
10005800273      7109     4976      5908     5908
                 9233     4976      5908    12697
...                        ...       ...      ...
2698443752596326 32774    5908       351     1117
                 105004   5908       351    15012
2698447711168309 4069     4716     12859     5897
                 8304    12859      4716     8594
                 30017    4716       586    12859

[64111 rows x 3 columns]

In [330]:

def f(sifid, mm):
    if mm:
        return sifid
    else:
        return pd.NA

ee['SIFIDLOW'] = ee.SIFIDLOW.combine(matchdfnumeric & low_mask, f)
ee['SIFIDHI'] = ee.SIFIDHI.combine(matchdfnumeric & hi_mask, f)

ee

SIFID SIFIDLOW SIFIDHI
int_id           cl_id                         
5710837346       20353    4976     6856    <NA>
                 20860    6856     4976    <NA>
                 52803    4976     <NA>    6856
10005800273      7109     4976     5908    5908
                 9233     4976     5908    <NA>
...                        ...      ...     ...
2698443752596326 32774    5908      351    <NA>
                 105004   5908      351    <NA>
2698447711168309 4069     4716    12859    <NA>
                 8304    12859     4716    <NA>
                 30017    4716     <NA>   12859

[64111 rows x 3 columns]

In [ ]:
centerlines_by_sifid = pd.concat(
    (centerlines_by_SIFID_SIFIDLOW, centerlines_by_SIFID_SIFIDHI),
    axis=1, keys=['SIFIDxLOW', 'SIFIDxHI'])

centerlines_by_sifid

def flip(index):
    return index[1], index[0]

def get(from_):
    def getter(index):
        try:
            match = from_.loc[index]
        except KeyError:
            match = pd.NA
        return match
    return getter 


cl_SSHI_index = centerlines_by_SIFID_SIFIDHI.index
rev = cl_SSHI_index.to_series().map(flip)
rev.transform({'low_match': get(centerlines_by_SIFID_SIFIDLOW),
               'hi_match': get(centerlines_by_SIFID_SIFIDHI)})

pd.concat((centerlines_by_SIFID_SIFIDLOW.index.to_series(),
           centerlines_by_SIFID_SIFIDHI.index.to_series()),
             axis=1, keys=['CLxSSLOW', 'CLxSSHI'])

def g(sifs):
    SIFID, LOW, HI = sifs
    hi_index = (HI, SIFID)
    low_index = (LOW, SIFID)



    try:
        hi_match = centerlines_by_SIFID_SIFIDHI.loc[hi_index]
    except KeyError:
        hi_match = None 

    try:
        match_sslow = centerlines_by_SIFID_SIFIDLOW[hi_index]
    except KeyError:
        if hi_match is None:
            hi_match = pd.NA
    else:
        if hi_match is None:
            hi_match = match_sslow
        else:
            hi_match = hi_match.union(match_sslow)
    return hi_match


,SIFIDLOW,SIFIDHI
OBJECTID,,
1,"(1550, 8665)","(8594, 8665)"
2,"(2854, 5926)","(2470, 5926)"
3,"(6487, 473)","(10596, 473)"
4,"(13394, 2442)","(4702, 2442)"
5,"(4162, 4573)","(8594, 4573)"
...,...,...
180232,"(8594, 15329)","(15331, 15329)"
180233,"(15610, 15611)","(15642, 15611)"
180234,"(15642, 15611)","(15610, 15611)"


In [ ]:
def get(index):
    try:
        hi_match = centerlines_by_SIFID_SIFIDHI.loc[index]
    except KeyError:
        hi_match = None 

    try:
        low_match = centerlines_by_SIFID_SIFIDLOW.loc[index]
    except KeyError:
        if hi_match is None:
            return pd.NA 
        else:
            return hi_match.to_list()
    else:
        if hi_match is None:
            return low_match.to_list()
        else:
            return hi_match.union(low_match).to_list()

def h(s):
    return s.combine(centerlines.SIFID, lambda x, y:(x, y)).apply(get)


centerlines[['SIFIDLOW', 'SIFIDHI']].apply(h)



,SIFIDLOW,SIFIDHI
OBJECTID,,
1,[6013],<NA>
2,"[3753, 23064]","[28081, 32011]"
3,"[20485, 24494, 27237]",[3057]
4,"[84286, 84288]",[84285]
5,[18357],<NA>
...,...,...
180232,<NA>,[145324]
180233,[180235],[180236]
180234,[180236],[180235]


In [410]:
centerlines.index.to_list()


[1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 20,
 21,
 22,
 23,
 25,
 30,
 31,
 32,
 33,
 35,
 37,
 38,
 39,
 40,
 46,
 47,
 48,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 62,
 63,
 64,
 68,
 71,
 72,
 73,
 75,
 76,
 77,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 130,
 131,
 132,
 133,
 135,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 162,
 167,
 168,
 169,
 170,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 186,
 187,
 188,
 189,
 190,
 191,
 192,
 193,
 194,
 195,
 196,
 197,
 198,
 199,
 200,
 201,
 202,
 203,
 204,
 205,
 206,
 209,
 218,
 219,
 220,
 221,
 224,
 225,
 226,
 227,
 228,
 229,
 231,
 232,
 233,
 234,
 236,
 237,
 239,
 242,
 243,
 244,
 245,

In [ ]:
ee['intersection_point'] = GEO_map.intersection_point
ee["GEO_close"] = GEO_map.GEO_close

ee

In [271]:
(ee[ee.SIFIDHI.notna() & ee.SIFIDLOW.notna()] == ee[ee.SIFIDLOW == ee.SIFIDHI]).all().all()
# -> True. This means that in all cases where both SIFIDHI and LOW match, the SIFIDs are equal
# Road loops and such as, I guess.

CLG = centerlines.groupby('SIFID')[['GEO_start', 'GEO_end']].apply(lambda x:x)
CLG



GEO_start  \
SIFID OBJECTID                                             
1     2398       [-85.84448159320799, 38.20137533915249]   
      3506       [-85.70968827136649, 38.12813112397158]   
      6313       [-85.89531203661909, 38.14401468934693]   
      8356       [-85.84798778460241, 38.17734130223805]   
      8683      [-85.84280313341836, 38.201214934112464]   
...                                                  ...   
15638 179908      [-85.60731013837355, 38.1857501107605]   
15639 179911     [-85.85171970535131, 38.20031227634193]   
15640 180227      [-85.5778646682938, 38.31733391246771]   
15641 180229    [-85.54618848348093, 38.084581472122935]   
15642 180236    [-85.45174798320075, 38.270920967505994]   

                                                 GEO_end  
SIFID OBJECTID                                            
1     2398       [-85.84498831396351, 38.20004206388769]  
      3506       [-85.70951084235313, 38.12868880056472]  
      6313       [-85.89203421085367, 38.14392222397775]  
      8356       [-85.84881288521419, 38.17714727759056]  
      8683       [-85.84448159320799, 38.20137533915249]  
...                                                  ...  
15638 179908    [-85.60609105189127, 38.185491770938995]  
15639 179911     [-85.85181439690278, 38.20435070734935]  
15640 180227     [-85.57028952405062, 38.31993204916915]  
15641 180229     [-85.54604439062057, 38.08411166310706]  
15642 180236     [-85.45058838480479, 38.27180114613091]  

[34215 rows x 2 columns]

In [277]:

dist = central_angle

def cg(s):
    LOW = s.SIFIDLOW
    HI = s.SIFIDHI
    if pd.isna(LOW):
        sifid = HI
    elif pd.isna(HI):
        sifid = LOW
    else:
        assert HI == LOW
        sifid = LOW # arbitrary
    int_match = s.GEO_close
    try:
        cl_geos = CLG.loc[sifid]#.map(lambda p:dist(int_match, p))
    except KeyError:
        return
    else:
        return type(cl_geos)


#ee.apply(cg, axis=1)

ee



SIFIDLOW SIFIDHI  \
int_id           cl_id                     
5710837346       20353      6856    <NA>   
                 20860      4976    <NA>   
                 52803      <NA>    6856   
10005800273      7109       5908    5908   
                 9233       5908    <NA>   
...                          ...     ...   
2698443752596326 32774       351    <NA>   
                 105004      351    <NA>   
2698447711168309 4069      12859    <NA>   
                 8304       4716    <NA>   
                 30017      <NA>   12859   

                                                       GEO_close  
int_id           cl_id                                            
5710837346       20353   [-85.51044384084946, 38.20588660809318]  
                 20860   [-85.51044384084946, 38.20588660809318]  
                 52803   [-85.51044384084946, 38.20588660809318]  
10005800273      7109    [-85.52816882852979, 38.20037561255241]  
                 9233    [-85.52816882852979, 38.20037561255241]  
...                                                          ...  
2698443752596326 32774     [-85.5284541859341, 38.2171605272513]  
                 105004  [-85.52706694300736, 38.22235647012922]  
2698447711168309 4069    [-85.68299954058695, 38.13164510965587]  
                 8304    [-85.68299954058695, 38.13164510965587]  
                 30017   [-85.68299954058695, 38.13164510965587]  

[64111 rows x 3 columns]

In [223]:
c = centerlines.groupby('SIFID')[['GEO_start', 'GEO_end']].apply(lambda x:x)
#centerlines[["SIFID", 'GEO_start', 'GEO_end']].groupby("SIFID").apply(lambda x:x)
c

GEO_start  \
SIFID OBJECTID                                             
1     2398       [-85.84448159320799, 38.20137533915249]   
      3506       [-85.70968827136649, 38.12813112397158]   
      6313       [-85.89531203661909, 38.14401468934693]   
      8356       [-85.84798778460241, 38.17734130223805]   
      8683      [-85.84280313341836, 38.201214934112464]   
...                                                  ...   
15638 179908      [-85.60731013837355, 38.1857501107605]   
15639 179911     [-85.85171970535131, 38.20031227634193]   
15640 180227      [-85.5778646682938, 38.31733391246771]   
15641 180229    [-85.54618848348093, 38.084581472122935]   
15642 180236    [-85.45174798320075, 38.270920967505994]   

                                                 GEO_end  
SIFID OBJECTID                                            
1     2398       [-85.84498831396351, 38.20004206388769]  
      3506       [-85.70951084235313, 38.12868880056472]  
      6313       [-85.89203421085367, 38.14392222397775]  
      8356       [-85.84881288521419, 38.17714727759056]  
      8683       [-85.84448159320799, 38.20137533915249]  
...                                                  ...  
15638 179908    [-85.60609105189127, 38.185491770938995]  
15639 179911     [-85.85181439690278, 38.20435070734935]  
15640 180227     [-85.57028952405062, 38.31993204916915]  
15641 180229     [-85.54604439062057, 38.08411166310706]  
15642 180236     [-85.45058838480479, 38.27180114613091]  

[34215 rows x 2 columns]

In [ ]:
def f(sifid, mm):
    if mm:
        return sifid
    else:
        return pd.NA
    
def g(sifid):
    try:
        geos = c.loc[sifid]
    except KeyError:
        return pd.NA
    else:
        ...


sl = ee.SIFIDLOW.combine(matchdfnumeric & low_mask, f)#.apply(g)


int_id            cl_id 
5710837346        20353      6856
                  20860      4976
                  52803      <NA>
10005800273       7109       5908
                  9233       5908
                            ...  
2698443752596326  32774       351
                  105004      351
2698447711168309  4069      12859
                  8304       4716
                  30017      <NA>
Length: 64111, dtype: object

In [293]:
sl.iloc[1]

,GEO_start,GEO_end
OBJECTID,,
7109,"[-85.52841872335158, 38.200360349057064]","[-85.52816882852979, 38.20037561255241]"
9233,"[-85.52816882852979, 38.20037561255241]","[-85.52154597701633, 38.20245302533696]"
14678,"[-85.50187141303726, 38.206105324500015]","[-85.49945278036108, 38.20743661819182]"
20166,"[-85.53836064009367, 38.200600417756]","[-85.531149709042, 38.2005045068783]"
20353,"[-85.51044384084946, 38.20588660809318]","[-85.50187141303726, 38.206105324500015]"
31468,"[-85.531149709042, 38.2005045068783]","[-85.52841872335158, 38.200360349057064]"
52803,"[-85.51425148406123, 38.20488285764657]","[-85.51044384084946, 38.20588660809318]"
52804,"[-85.52154597701633, 38.20245302533696]","[-85.51425148406123, 38.20488285764657]"
90023,"[-85.49041133329159, 38.204811157321686]","[-85.48494532437061, 38.2079572283897]"


In [ ]:
# -> make  SIFdf

# cl_dat = centerlines[['SIFID', 'SIFIDLOW', 'SIFIDHI']]

# def gc(ci):
#     return cl_dat.loc[ci]

# sifm = matchdfnumeric.index.to_frame().cl_id.apply(gc)
# sifm

# any(matchdfnumeric == 0b0011) # none
# any(matchdfnumeric == 0b0000) # false # none of these either

# data = {"int_match": matchdfnumeric & 0b0011,
#         "cl_sifid": sifm.SIFID,
#         "low_match": sifm.SIFIDLOW[(matchdfnumeric & low_mask) != 0],
#         "hi_match": sifm.SIFIDHI[(matchdfnumeric & hi_mask) != 0]}


# #dd = pd.concat((int_ser, sifm.SIFID, low_ser, hi_ser), axis=1, keys=['int_match', 'cl_sifid', 'low_match', 'hi_match']).convert_dtypes()

# SIFdf = pd.DataFrame.from_dict(data).convert_dtypes()
# SIFdf

In [61]:
# make multi index coumns helper

# def ix(columns, *labels):
#     out = list()
#     columns = iter(columns)
#     for label, column in zip(labels, columns):
#         out.append((label, column))
#     else:
#         # default = last value of label
#         for column in columns:
#             out.append((label, column))
#     return pd.MultiIndex.from_tuples(out)

# ix(NAME_df.columns, 'hello')

# NAME_df.columns=ix(NAME_df.columns, 'hello')
# NAME_df

In [71]:

from common.county_geometry import *

iw = matchdfnumeric.index.to_frame()

intersections_GEOMETRY = intersections.GEOMETRY
intersections_state_grid_GEO = intersections.state_grid_GEO

centerlines_GEOLOW = centerlines.GEOLOW
centerlines_GEOHI = centerlines.GEOHI



In [ ]:
hav = haversine_distance_ft

intersection_coordinates = iw.int_id.apply(lambda int_id:intersections_GEOMETRY.at[int_id])
intersection_state_grid_conv = iw.int_id.apply(lambda int_id:intersections_state_grid_GEO.at[int_id])

# TODO ...

In [ ]:
ids = mdf.index.to_frame(name=['intersection_ids',"centerline_ids"])

int_long_lat = ids.intersection_ids.apply(lambda intid: intersections_GEO.at[intid])
int_state_grid = ids.intersection_ids.apply(lambda intid: intersections_state_grid.at[intid])

hi_points = ids.centerline_ids.apply(lambda cl_id: centerlines_GEOHI.at[cl_id])
low_points = ids.centerline_ids.apply(lambda cl_id: centerlines_GEOLOW.at[cl_id])

int_long_lat.combine(hi_points, central_angle)
int_state_grid.combine(hi_points, central_angle)

int_long_lat.combine(hi_points, central_angle)
int_state_grid.combine(hi_points, central_angle)

int_id           cl_id 
589991067518338  10250     2.542220e-05
                 28655     4.687739e-05
590506463593858  10250     2.816241e-05
                 28655     4.542790e-05
409323268213029  26919     3.775007e-05
                               ...     
847134907955010  174788    1.436194e-07
847207302035270  175429    1.431915e-07
847212217628486  175434    1.431968e-07
847246577432390  175435    1.431938e-07
847250872399689  175437    1.431925e-07
Length: 64111, dtype: float64

In [ ]:

hi_dist = (int_points - hi_points).apply(euclidean_distance)

low_dist = (int_points - low_points).apply(euclidean_distance)


In [ ]:

def encode_closer_point(difference):
    if difference < 0: # hi_dist < low_dist
        return 'hi'
    elif difference > 0: # hi_dist > low_dist
        return 'low'
    elif difference == 0: # hi_dist == low_dist
        return 'both'
    else:
        return pd.NA

dist_diff = (hi_dist - low_dist)
geo_end = dist_diff.apply(encode_closer_point)


geo_end.hasnans # -> False.
# checking something;
(geo_end[geo_end != 'low'] == geo_end[(geo_end == 'hi') | (geo_end == 'both')]).all() # -> True

In [ ]:
close_points = pd.concat((
    hi_points[geo_end != 'low'], 
    low_points[geo_end == 'low']))

close_distance = pd.concat((
    hi_dist[geo_end != 'low'],
    low_dist[geo_end == 'low']))

far_points = pd.concat((
    hi_points[geo_end == 'low'],
    low_points[geo_end == 'hi']))

far_distance = pd.concat((
    hi_dist[geo_end == 'low'],
    low_dist[geo_end == 'hi']))


working = pd.DataFrame(pd.Series(int_points, name='int_point'))
working['geo_end'] = geo_end
working['close_point'] = close_points
working['far_point'] = far_points
working['close_dist'] = close_distance
working['far_dist'] = far_distance
working['dist_diff'] = dist_diff.abs()

working.index.set_names(('int_id', 'cl_id'), inplace=True)
geo_data = working

geo_data

int_point geo_end  \
int_id          cl_id                                                      
589991067518338 10250    [-85.89203421085367, 38.14392222397775]     low   
                28655    [-85.89203421085367, 38.14392222397775]     low   
590506463593858 10250   [-85.89205718950319, 38.143226353738676]     low   
                28655   [-85.89205718950319, 38.143226353738676]     low   
409323268213029 26919   [-85.83013514539758, 38.213609743764856]     low   
...                                                          ...     ...   
847134907955010 174788   [-85.49669357922623, 38.29070651405069]      hi   
847207302035270 175429   [-85.55862850424626, 38.12636761108555]      hi   
847212217628486 175434   [-85.55884906334812, 38.12802620488927]      hi   
847246577432390 175435    [-85.55854188749127, 38.1271646809743]      hi   
847250872399689 175437   [-85.55753279215419, 38.12710321087191]      hi   

                                                     close_point  \
int_id          cl_id                                              
589991067518338 10250    [-85.89203421085367, 38.14392222397775]   
                28655   [-85.89205718950319, 38.143226353738676]   
590506463593858 10250    [-85.89203421085367, 38.14392222397775]   
                28655   [-85.89205718950319, 38.143226353738676]   
409323268213029 26919   [-85.83013514539758, 38.213609743764856]   
...                                                          ...   
847134907955010 174788   [-85.49669357922623, 38.29070651405069]   
847207302035270 175429   [-85.55862850424626, 38.12636761108555]   
847212217628486 175434   [-85.55884906334812, 38.12802620488927]   
847246577432390 175435    [-85.55854188749127, 38.1271646809743]   
847250872399689 175437   [-85.55753279215419, 38.12710321087191]   

                                                       far_point  close_dist  \
int_id          cl_id                                                          
589991067518338 10250    [-85.89017790736897, 38.14387530855226]    0.000000   
                28655    [-85.89535933539815, 38.14331916493513]    0.000696   
590506463593858 10250    [-85.89017790736897, 38.14387530855226]    0.000696   
                28655    [-85.89535933539815, 38.14331916493513]    0.000000   
409323268213029 26919    [-85.83201078511965, 38.21202291188871]    0.000000   
...                                                          ...         ...   
847134907955010 174788  [-85.49758979847759, 38.290228130801005]    0.000000   
847207302035270 175429    [-85.55854188749127, 38.1271646809743]    0.000000   
847212217628486 175434   [-85.55899003824544, 38.12895721169443]    0.000000   
847246577432390 175435   [-85.55884906334812, 38.12802620488927]    0.000000   
847250872399689 175437  [-85.55664071193948, 38.127332178618765]    0.000000   

                        far_dist  dist_diff  
int_id          cl_id                        
589991067518338 10250   0.001857   0.001857  
                28655   0.003379   0.002683  
590506463593858 10250   0.001988   0.001292  
                28655   0.003303   0.003303  
409323268213029 26919   0.002457   0.002457  
...                          ...        ...  
847134907955010 174788  0.001016   0.001016  
847207302035270 175429  0.000802   0.000802  
847212217628486 175434  0.000942   0.000942  
847246577432390 175435  0.000915   0.000915  
847250872399689 175437  0.000921   0.000921  

[64111 rows x 7 columns]

In [ ]:

def get_intersection_names(intersection_id):
    in1 = intersections.at[intersection_id, 'FST_ROADNAME']
    in2 = intersections.at[intersection_id, 'SEC_ROADNAME']
    return in1, in2

names = centerlines.groupby("SIFID").ROADNAME.apply(set)#.apply(lambda x:len(x)==1).all() # true
sifid_names = names.apply(lambda x:x.pop())

def roadname_by_sifid(sifid):
    return sifid_names[sifid]

def get_names(int_id, cd_id):
    inames = get_intersection_names(intersection_id)
    cnames = roadname_by_sifid(cl_id)
    return {'intersection':inames, 'centerline':cnames}



In [ ]:
intids = working.index.to_frame().int_id
intids

int_id           cl_id 
589991067518338  10250     589991067518338
                 28655     589991067518338
590506463593858  10250     590506463593858
                 28655     590506463593858
409323268213029  26919     409323268213029
                                ...       
847134907955010  174788    847134907955010
847207302035270  175429    847207302035270
847212217628486  175434    847212217628486
847246577432390  175435    847246577432390
847250872399689  175437    847250872399689
Name: int_id, Length: 64111, dtype: int64

In [ ]:
working = geo_data[geo_data.close_dist >= .1]
working

iw = working.index.to_frame()
iw

def get_int_data(int_id):
    return intersections.loc[int_id][["FST_ROADNAME", "FST_SIFID", "SEC_ROADNAME", "SEC_SIFID"]]

def get_cl_data(cl_id):
    return centerlines.loc[cl_id][['ROADNAME', 'SIFID', 'CORE_CLASS']]

data = pd.concat((
iw.cl_id.transform({"centerline_data":get_cl_data}) ,
iw.int_id.transform({'intersection_data':get_int_data})
), axis=1)

data


def mmft(label, df):
    if isinstance(label, str):
        return pd.MultiIndex.from_tuples((label, col) for col in df.columns)
    else:
        return pd.MultiIndex.from_tuples((L, C) for L, C in zip(label, df.columns))
    


def dd(geodf):
    #ex = geodf[['int_point
    #out = list()
    iw = geodf.index.to_frame()
    cl_geo = geodf[['int_point', 'close_point', 'close_dist', 'geo_end']]
    cl_geo.columns = mmft(('int_geo', 'cl_geo', 'cl_geo', 'cl_geo'), cl_geo)

    cl_data = iw.cl_id.apply(get_cl_data)
    cl_data.columns = mmft('cl_data', cl_data)
    # TODO get hi/low sifid / roadname

    int_data = iw.int_id.apply(get_int_data)
    int_data.columns = mmft("int_data", int_data)

    out = [int_data, cl_geo, cl_data]
    return pd.concat(out, axis=1)
    
    int_data = iw.int_id.apply(get_int_data)
    int_info = pd.concat((int_data, geodf.int_point), axis=1)
    int_info.columns = pd.MultiIndex.from_tuples(('int_data', col) for col in int_info.columns)
    
    return pd.concat((int_info, cl_info), axis=1).sort_index()

#    pd.MultiIndex.from_tuples([('cl_data', col) for col in c.columns])
#c.columns = pd.MultiIndex.from_tuples([('cl_data', col) for col in c.columns])
#c

see = dd(working)

#see[~see.int_data.FST_ROADNAME.str.contains('841')]

see

int_data                                    \
                          FST_ROADNAME FST_SIFID  SEC_ROADNAME SEC_SIFID   
int_id           cl_id                                                     
590467808889140  29463  NO STREET NAME         1   CANE RUN RD       906   
38521867941168   6218     COLUMBIA AVE      1245  KENTUCKY AVE      3356   
589634891494704  21710    COLUMBIA AVE      1245  KENTUCKY AVE      3356   
706470196341142  903       KY-841 RAMP     14021        KY 841     14195   
                 18413     KY-841 RAMP     14021        KY 841     14195   
...                                ...       ...           ...       ...   
654251992213784  6590           KY 841     14195   KY-841 RAMP     14021   
                 7706           KY 841     14195   KY-841 RAMP     14021   
2696491761592600 6590           KY 841     14195   KY-841 RAMP     14021   
                 7706           KY 841     14195   KY-841 RAMP     14021   
                 11399          KY 841     14195   KY-841 RAMP     14021   

                                                         int_geo  \
                                                       int_point   
int_id           cl_id                                             
590467808889140  29463   [-85.89679114783614, 38.14336979587274]   
38521867941168   6218    [-85.61845036330493, 38.26000799417955]   
589634891494704  21710  [-85.84807001201702, 38.150808026597076]   
706470196341142  903    [-85.70246758147974, 38.115233764331414]   
                 18413  [-85.70246758147974, 38.115233764331414]   
...                                                          ...   
654251992213784  6590   [-85.87003129510553, 38.091380651748494]   
                 7706   [-85.87003129510553, 38.091380651748494]   
2696491761592600 6590    [-85.87691759786703, 38.09294723096209]   
                 7706    [-85.87691759786703, 38.09294723096209]   
                 11399   [-85.87691759786703, 38.09294723096209]   

                                                          cl_geo             \
                                                     close_point close_dist   
int_id           cl_id                                                        
590467808889140  29463   [-85.8175947343509, 38.218723744753156]   0.109317   
38521867941168   6218   [-85.84807001201702, 38.150808026597076]   0.254263   
589634891494704  21710   [-85.61845036330493, 38.26000799417955]   0.254263   
706470196341142  903     [-85.8189716474408, 38.103260224945146]   0.117118   
                 18413  [-85.87003129510553, 38.091380651748494]   0.169253   
...                                                          ...        ...   
654251992213784  6590   [-85.75479831645234, 38.117411563113556]   0.118137   
                 7706    [-85.74797696339039, 38.11669772522921]   0.124652   
2696491761592600 6590   [-85.75479831645234, 38.117411563113556]   0.124546   
                 7706    [-85.74797696339039, 38.11669772522921]   0.131110   
                 11399   [-85.77549954907987, 38.11946730096766]   0.104828   

                                     cl_data                         
                       geo_end      ROADNAME  SIFID      CORE_CLASS  
int_id           cl_id                                               
590467808889140  29463      hi       NO NAME      1           LOCAL  
38521867941168   6218      low  COLUMBIA AVE   1245           LOCAL  
589634891494704  21710     low  COLUMBIA AVE   1245           LOCAL  
706470196341142  903       low   KY-841 RAMP  14021  MAJOR ARTERIAL  
                 18413     low   KY-841 RAMP  14021  MAJOR ARTERIAL  
...                        ...           ...    ...             ...  
654251992213784  6590       hi   KY-841 RAMP  14021  MAJOR ARTERIAL  
                 7706      low   KY-841 RAMP  14021  MAJOR ARTERIAL  
2696491761592600 6590       hi   KY-841 RAMP  14021  MAJOR ARTERIAL  
                 7706      low   KY-841 RAMP  14021  MAJOR ARTERI

In [ ]:
s= working.index.get_level_values('cl_id')

centerlines.loc[s].ROADNAME.value_counts()

ROADNAME
KY-841 RAMP     39
KY 841          36
COLUMBIA AVE     4
CANE RUN RD      4
KENTUCKY AVE     4
NO NAME          2
Name: count, dtype: int64

In [ ]:
mi = intersections.loc[working.index.get_level_values('int_id')]

# KY 841 / I 265 / Watterson Expwy and ramps
mi[(mi.FST_ROADNAME.str.contains("841") & mi.SEC_ROADNAME.str.contains("841"))]

# other weird matches
wm = mi[~(mi.FST_ROADNAME.str.contains("841") & mi.SEC_ROADNAME.str.contains("841"))]

working.loc[wm.index, :]
wm
mi


,FST_ROADNAME,SEC_ROADNAME,SIFCODE1,SIFCODE2,FST_SIFID,SEC_SIFID,GEOMETRY,state_grid_GEO
int_id,,,,,,,,
590467808889140,NO STREET NAME,CANE RUN RD,0000,0934,1,906,"[-85.89679114783614, 38.14336979587274]","[-85.89673981756692, 38.143358031236794]"
38521867941168,COLUMBIA AVE,KENTUCKY AVE,1241,3530,1245,3356,"[-85.61845036330493, 38.26000799417955]","[-85.61844551689906, 38.260000695318304]"
589634891494704,COLUMBIA AVE,KENTUCKY AVE,1241,3530,1245,3356,"[-85.84807001201702, 38.150808026597076]","[-85.84805632815548, 38.15080051575182]"
706470196341142,KY-841 RAMP,KY 841,E918,E996,14021,14195,"[-85.70246758147974, 38.115233764331414]","[-85.70246272131506, 38.11522649717792]"
706470196341142,KY-841 RAMP,KY 841,E918,E996,14021,14195,"[-85.70246758147974, 38.115233764331414]","[-85.70246272131506, 38.11522649717792]"
...,...,...,...,...,...,...,...,...
654251992213784,KY 841,KY-841 RAMP,E996,E918,14195,14021,"[-85.87003129510553, 38.091380651748494]","[-85.87002638784153, 38.09137339695274]"
654251992213784,KY 841,KY-841 RAMP,E996,E918,14195,14021,"[-85.87003129510553, 38.091380651748494]","[-85.87002638784153, 38.09137339695274]"
2696491761592600,KY 841,KY-841 RAMP,E996,E918,14195,14021,"[-85.87691759786703, 38.09294723096209]","[-85.87691268847966, 38.09293997618528]"


In [ ]:


    

#display(
#info[['int_point', 'close_point', 'close_dist', 'geo_end', ]])



wmm = data[~data.intersection_data.FST_ROADNAME.str.contains('841')]

display(
working.loc[wmm.index][['int_point', 'close_point', 'close_dist', 'geo_end', ]],
wmm)


,,int_point,close_point,close_dist,geo_end
int_id,cl_id,,,,
590467808889140,29463,"[-85.89679114783614, 38.14336979587274]","[-85.8175947343509, 38.218723744753156]",0.109317,hi
38521867941168,6218,"[-85.61845036330493, 38.26000799417955]","[-85.84807001201702, 38.150808026597076]",0.254263,low
589634891494704,21710,"[-85.84807001201702, 38.150808026597076]","[-85.61845036330493, 38.26000799417955]",0.254263,low
352187318274356,31519,"[-85.81503148910512, 38.21843165728323]","[-85.89535933539815, 38.14331916493513]",0.109975,low
38521867941168,10508,"[-85.61845036330493, 38.26000799417955]","[-85.84620751970706, 38.15069748385449]",0.252630,low
589634891494704,11695,"[-85.84807001201702, 38.150808026597076]","[-85.61953362264914, 38.25940896160892]",0.253028,low
352187318274356,11220,"[-85.81503148910512, 38.21843165728323]","[-85.89679114783614, 38.14336979587274]",0.110991,low
590467808889140,26722,"[-85.89679114783614, 38.14336979587274]","[-85.81797210922643, 38.21544151873681]",0.106803,hi
38521867941168,2474,"[-85.61845036330493, 38.26000799417955]","[-85.84807001201702, 38.150808026597076]",0.254263,low


centerline_data                           \
                             ROADNAME SIFID         CORE_CLASS   
int_id          cl_id                                            
590467808889140 29463         NO NAME     1              LOCAL   
38521867941168  6218     COLUMBIA AVE  1245              LOCAL   
589634891494704 21710    COLUMBIA AVE  1245              LOCAL   
352187318274356 31519         NO NAME     1              LOCAL   
38521867941168  10508    COLUMBIA AVE  1245              LOCAL   
589634891494704 11695    COLUMBIA AVE  1245              LOCAL   
352187318274356 11220     CANE RUN RD   906  PRIMARY COLLECTOR   
590467808889140 26722     CANE RUN RD   906     MINOR ARTERIAL   
38521867941168  2474     KENTUCKY AVE  3356              LOCAL   
589634891494704 29055    KENTUCKY AVE  3356              LOCAL   
352187318274356 32589     CANE RUN RD   906  PRIMARY COLLECTOR   
590467808889140 5361      CANE RUN RD   906     MINOR ARTERIAL   
38521867941168  10465    KENTUCKY AVE  3356              LOCAL   
589634891494704 5523     KENTUCKY AVE  3356              LOCAL   

                      intersection_data                                    
                           FST_ROADNAME FST_SIFID  SEC_ROADNAME SEC_SIFID  
int_id          cl_id                                                      
590467808889140 29463    NO STREET NAME         1   CANE RUN RD       906  
38521867941168  6218       COLUMBIA AVE      1245  KENTUCKY AVE      3356  
589634891494704 21710      COLUMBIA AVE      1245  KENTUCKY AVE      3356  
352187318274356 31519    NO STREET NAME         1   CANE RUN RD       906  
38521867941168  10508      COLUMBIA AVE      1245  KENTUCKY AVE      3356  
589634891494704 11695      COLUMBIA AVE      1245  KENTUCKY AVE      3356  
352187318274356 11220    NO STREET NAME         1   CANE RUN RD       906  
590467808889140 26722    NO STREET NAME         1   CANE RUN RD       906  
38521867941168  2474       COLUMBIA AVE      1245  KENTUCKY AVE      3356  
589634891494704 29055      COLUMBIA AVE      1245  KENTUCKY AVE      3356  
352187318274356 32589    NO STREET NAME         1   CANE RUN RD       906  
590467808889140 5361     NO STREET NAME         1   CANE RUN RD       906  
38521867941168  10465      COLUMBIA AVE      1245  KENTUCKY AVE      3356  
589634891494704 5523       COLUMBIA AVE      1245  KENTUCKY AVE      3356

In [ ]:
centerlines[centerlines.ROADNAME.str.contains("KENTUCKY AVE")]
centerlines[(centerlines.SIFID == 3356) & ((centerlines.SIFIDLOW == 1245) | (centerlines.SIFIDHI == 1245))]

#wm = wm.groupby(by="FST_SIFID").apply(lambda x:x)


,ROADNAME,SIFID,SIFCODE,low_cross_ROADNAME,SIFIDLOW,LOCROSSSIF,hi_cross_ROADNAME,SIFIDHI,HICROSSSIF,CORE_CLASS,GEOLOW,GEOHI
OBJECTID,,,,,,,,,,,,
2474,KENTUCKY AVE,3356,3530,COLUMBIA AVE,1245,1241,PRINCETON AVE,4725,5155,LOCAL,"[-85.84807001201702, 38.150808026597076]","[-85.84822119095742, 38.149025980797525]"
5523,KENTUCKY AVE,3356,3530,FLORIDA AVE,2249,2251,COLUMBIA AVE,1245,1241,LOCAL,"[-85.6148949649379, 38.25574321445772]","[-85.61845036330493, 38.26000799417955]"
10465,KENTUCKY AVE,3356,3530,DEAD END,8594,9811,COLUMBIA AVE,1245,1241,LOCAL,"[-85.84789878936502, 38.1526162694713]","[-85.84807001201702, 38.150808026597076]"
29055,KENTUCKY AVE,3356,3530,COLUMBIA AVE,1245,1241,DEAD END,8594,9811,LOCAL,"[-85.61845036330493, 38.26000799417955]","[-85.62012324704497, 38.25992290121035]"


In [ ]:
def convert_geo(geo):
    if len(geo) == 2:
        long, lat = geo
        return (lat, long)
    else:
        return [(lat, long) for long, lat in geo]

def swap_point(point):
    x, y = point
    return (y, x)

swap_point((1,2))
        

(2, 1)

In [ ]:
# working = mdf.index.to_series()


# def get_geo(row):
#     int_id, cl_id = row
#     out = dict() # tried Series. Took too long. Dict is very fast ( .4 sec)
#     out['intx_geo'] = intx_geo = intersections_GEO.at[int_id]
#     out['cl_geo_low'] = centerlines_GEOLOW.at[cl_id]
#     out['cl_geo_hi'] = centerlines_GEOHI.at[cl_id]
#     return out

# working = pd.DataFrame(mdf.index.to_series().apply(get_geo).to_list(), index=mdf.index)
# hi_dist = (working.intx_geo - working.cl_geo_hi).apply(euclidean_distance)
# low_dist = (working.intx_geo - working.cl_geo_low).apply(euclidean_distance)

# def find_close_closer_point(x):
#     if x < 0:
#         return 'hi'
#     elif x > 0:
#         return 'low'
#     elif x == 0:
#         return 'both'
#     else:
#         return pd.NA

# working['geo_end'] = (hi_dist - low_dist).apply(find_close_closer_point)
# working

# def gp(row):
#     code = row.geo_end
#     if code == 'low':
#         return (row.cl_geo_low, row.cl_geo_hi)
#     elif code == 'hi':
#         return (row.cl_geo_hi,row.cl_geo_low)
#     elif code == 'both':
#         return (row.cl_geo_hi, None)
#     else:
#         return (None, None)

# ee = pd.DataFrame(working.apply(gp, axis=1).to_list(), index=working.index, columns=['close_point', 'far_point'])
# pd.concat((working, ee), axis=1)


In [ ]:
# old versions of code
#     lowdist = euclidean_distance(intx_geo - cl_geo_low)
#     hidist = euclidean_distance(intx_geo - cl_geo_hi)
#     if lowdist < hidist:
#         geo_close = 'low'
#         closer_point = cl_geo_low
#         farther_point = cl_geo_hi
#         close_dist = lowdist
#         far_dist = hidist
#     elif lowdist > hidist:
#         geo_close = 'hi'
#         closer_point = cl_geo_hi
#         farther_point = cl_geo_low
#         close_dist = lowdist
#         far_dist = hidist
#     elif lowdist == hidist:
#         geo_close = 'both'
#         #assert cl_geo_low == cl_geo_hi
#         closer_point = cl_geo_low
#         farther_point = pd.NA
#         close_dist = lowdist
#         far_dist = hidist

#     out['geo_close'] = geo_close
#     out['close_point'] = closer_point
#     out['far_point'] = farther_point
#     out['close_dist'] = close_dist
#     out['far_dist'] = far_dist

#     return out


# working = pd.DataFrame(mdf.index.to_series().apply(find_close_closer_point).to_list(), index=mdf.index)
# #centerlines.loc[working[working.closer_code == 'both'].index.get_level_values(1)] # closer_code = 'both'


In [ ]:

diffs = (working['close_dist'] - working['far_dist']).abs()
# some values are zero
# drop these to make finding min easier
dd = diffs.drop(diffs[diffs == 0].index)


dd.min()

np.float64(0.0005881797990768545)

In [ ]:
# Problematic numbers?

#intersections.loc[319438198302071]
#centerlines.loc[22602]

#centerlines.loc[[84838, 31468]]
#(-85.5284187234, 38.2003603491)
#intersections.loc[[14300767569, 10005800273]]
# Problematic numbers?


#centerlines.loc[[13164, 24805, 26558]],
#intersections.loc[389674641274662])

#low_match_1[low_match_1 == 389674641274662]

# cl, ix =88673, {334354632500584, 726302145904921}


# 5 have 3 matches

# removing ramps from centerlines removed 2 of these

# 11577    {318119662229912, 318128252164504, 31811536726...
# Arthur St. and I 65 RAMP

# 79773    {731803473713560, 844908543362584, 84487847859...
# I 65 ramp

# 80073    {844438991051160, 731813464524312, 73181635861...
# I 65 ramp

# 31660    {582161378284545, 582191443055617, 61834218278...

#582161378284545	AUTUMN WAY	244	SUMMERTIME PKY	8232	(-85.8735529704, 38.0723544119)
#582191443055617	AUTUMN WAY	244	SUMMERTIME PKY	8232	(-85.8741780941, 38.0721878176)
#618342182786049	AUTUMN WAY	244	SUMMERTIME PKY	8232	(-85.873007987, 38.0733107806)

# 19920    {731744745003415, 722828392896919, 72287563753...
# I 65 RAMP



# 1 has 4 matches -> 
# removing EXPRESSWAYS from centerlines dealt with this one.

# 79772 : {731803473713560, 731813464524312, 844878478591512, 844908543362584}
# I 65 NORTH x I 65 RAMP
# no more from there 


# potential problematic numbers

#set(hi_match.keys()).intersection(hi_match2.keys()) # empty : good news
#  14288: ('start', 360168983520312, 8.767631726955189e-06),

#display(
##centerlines.loc[14288],
#intersections.loc[[360168983520312, 360748804105272]]
#

#centerlines.loc[[52803, 20860]]
#centerlines[centerlines.SIFIDLOW == 8594]
#centerlines.loc[[ 1498,  2541,  2610,  3823,  3882,  5039,  6722,  7769,  8692, 11961,
#       14175, 15423, 19704, 23933, 24872, 27737, 29258, 30870]] # I 265 RAMP.
#intersections.loc[847265632743817]
#intersections.loc[722828392896919]
#centerlines.loc[19920]
#ICpairs.loc[6146]

In [ ]:
# roadways = centerlines.groupby('SIFID').groups
# intxn_by_fst_sifid = intersections.groupby('FST_SIFID').groups
# # intxn_by_sec_sifid = intersections.groupby('SEC_SIFID') # probably not necessary
#     # ... since all records will be accessed by iterating over the fst_sifid groupby

# #intersections.FST_SIFID.hasnans # == False this is good

# def map_intersection_ids_to_centerline_ids(intersection_sifid_groups=intxn_by_fst_sifid, centerline_sifid_groups=roadways):
#     found = dict()
#     notfound = list()
#     for sifid_1 in intersection_sifid_groups.keys():
#         roadway_ids = centerline_sifid_groups.get(sifid_1, None)
#         if roadway_ids is not None:
#             found[sifid_1] = roadway_ids # first sifid match to centerlines id
#         else:
#             notfound.append(sifid_1)
#     return found, notfound

# def map_intersections_to_centerlines(intersections=intersections, centerlines=centerlines):
#     mapping = pd.DataFrame(columns=['full_match', 'fst_match_only', 'sec_match_only'])
#     full_matches = pd.Series(name='full_match', dtype="O")
#     roadways = centerlines.groupby('SIFID').groups
#     intersections_by_first_sifid = intersections.groupby('FST_SIFID').groups
#     notfound = list()

#     for sifid_1, intersection_ids in intersections_by_first_sifid.items():
#         centerline_ids = roadways.get(sifid_1, None)
#         if centerline_ids is None:
#             notfound.extend(intersection_ids)
#         else:
#             fst_match = centerlines.loc[centerline_ids]
#             for intersection_id in intersection_ids:
#                 sifid_2 = intersections.at[intersection_id, 'SEC_SIFID']
#                 full_match = fst_match[(fst_match.SIFIDLOW == sifid_2) | (fst_match.SIFIDHI == sifid_2)]
#                 if full_match.empty:
#                     mapping.at[intersection_id, 'fst_match_only'] = True
#                 else:
#                     full_match = full_match.index.tolist()
#                     #print(full_match)
#                     full_matches.at[intersection_id] = full_match
        
#     if notfound:
#         centerline_sifids = roadways.keys()
#         for intersection_id, sifid_2 in intersections.loc[notfound]['SEC_SIFID'].items():
#             if sifid_2 in centerline_sifids:
#                 mapping.at[intersection_id, 'sec_match_only'] = True

#     return pd.concat((full_matches, mapping))


# mapping = map_intersections_to_centerlines()


In [ ]:
#mapping.isna().apply(any, axis=1).all()
# test that each row has at least one value filled -> Yes
#intersections.index.difference(mapping.index) # == Index([693211604576105], dtype='int64')

#display(intersections.loc[693211604576105]) -> 13554, 13555 fst, sec ids

#centerlines[centerlines.SIFID == 13555] # nope, nor 13554 
# probably just ignore this.

#mapping[mapping.full_match.notna()]
#mapping

In [ ]:
# mapping = pd.DataFrame(columns=['full_match', 'fst_match_only', 'sec_match_only'])
# roadways = centerlines.groupby('SIFID').groups
# intersections_by_first_sifid = intersections.groupby('FST_SIFID').groups
# notfound = list()

# for sifid_1, intersection_ids in intersections_by_first_sifid.items():
#     centerline_ids = roadways.get(sifid_1, None)
#     if centerline_ids is None:
#         notfound.extend(intersection_ids)
#     else:
#         fst_match = centerlines.loc[centerline_ids]
#         #print(fst_match)
#         for intersection_id in intersection_ids:
#             sifid_2 = intersections.at[intersection_id, 'SEC_SIFID']
#             full_match = fst_match[(fst_match.SIFIDLOW == sifid_2) | (fst_match.SIFIDHI == sifid_2)]



In [ ]:
centerlines[centerlines.SIFID==1178]
notfound = [624,626,689,903,2053,122932,139877,161371,161372,168074]

intersections[intersections.FST_SIFID == 1178]
nf1 = intersections[intersections.FST_SIFID.isin(notfound)]
nf2 = centerlines[centerlines.SIFID.isin(nf1.SEC_SIFID.values)] # looks to be all interstate ramps
# it makes sense that these would not be in the centerline data b/c I probably stripped them out at a some point
# you can't/shouldn't ride a bicycle on the ramps/interstate!

noramp = nf2[nf2.CORE_CLASS != "INTERSTATE RAMP"] # 205 ramps. We don't need these roadways.
#display(noramp)
#nf.groupby('CORE_CLASS').count()

#nf.groupby('SIFID').groups
#sifid_twos = intersections[intersections.FST_SIFID.isin(notfound)].SEC_SIFID.values#.groupby('SEC_SIFID').groups
#for sifid_2 in sifid_twos:
#    i = centerlines[centerlines.SIFID == sifid_2].index.tolist()
#    if not i:
#        print(sifid_2)

display(nf1, nf2)

,FST_ROADNAME,SEC_ROADNAME,SIFCODE1,SIFCODE2,FST_SIFID,SEC_SIFID,GEOMETRY,state_grid_GEO
INTID,,,,,,,,
25314647488608,BRITTANY VALLEY RD,LIME KILN LN,0692,3860,689,3636,"[-85.63739842826769, 38.29902665283227]","[-85.63739357340009, 38.29901934737176]"
25533690815846,BRITTANY VALLEY RD,GLENVIEW AVE,0692,2566,689,2525,"[-85.6434753101308, 38.2964164239941]","[-85.64347045368233, 38.296409119316834]"
285078996590724,COFFEE TREE LN,COFFEE TREE PL,2053,2084,2053,2101,"[-85.53694581749606, 38.18196699123739]","[-85.53692786727781, 38.18195926142264]"
374478301589619,BOWLES AVE,WEBSTER ST,0624,7073,626,6361,"[-85.72680382327565, 38.25712141735226]","[-85.72679894542387, 38.25711412408772]"
374942158031000,BOWLES AVE,CABEL ST,0624,0898,626,875,"[-85.72832394001115, 38.256469323450425]","[-85.72831906176425, 38.256462030381684]"
414945534616712,CANDOR AVE,GARRS LN,0931,2488,903,2460,"[-85.81636719262018, 38.189799864851885]","[-85.81636229366431, 38.189792588666876]"
422723720394881,CANDOR AVE,LINHERK AVE,0931,3881,903,3654,"[-85.8169993940186, 38.18700350303334]","[-85.81699449508736, 38.18699622741413]"
432752417703459,BOWIE CT,BOWIE DR,0622,0623,624,625,"[-85.70260626299203, 38.14657969155578]","[-85.70260140045856, 38.1465724183819]"


,ROADNAME,SIFID,SIFCODE,low_cross_ROADNAME,SIFIDLOW,LOCROSSSIF,hi_cross_ROADNAME,SIFIDHI,HICROSSSIF,CORE_CLASS,GEOLOW,GEOHI
OBJECTID,,,,,,,,,,,,
3034,LIME KILN LN,3636,3860,LANSDOWNE AVE,3536,3743,RIVER KNOLLS DR,5040,5534,PRIMARY COLLECTOR,"[-85.64473369572579, 38.31284696929362]","[-85.64555838680533, 38.313566956021305]"
3125,WEBSTER ST,6361,7073,STORY AVE,13416,6183,BOWLES AVE,626,0624,LOCAL,"[-85.72647299204654, 38.25665296892628]","[-85.72680382327565, 38.25712141735226]"
3474,GARRS LN,2460,2488,CANDOR AVE,903,0931,NORTH LN,4265,4593,LOCAL,"[-85.81636719262018, 38.189799864851885]","[-85.81741529074715, 38.18987101606344]"
4249,GLENVIEW AVE,2525,2566,DUNRAVEN DR,1696,1726,DEAD END,8594,9811,LOCAL,"[-85.64201434345283, 38.29437496714594]","[-85.64219805634245, 38.29457377056122]"
4476,BOWIE DR,625,0623,BOWIE CT,624,0622,HICKOCK DR,13326,2913,LOCAL,"[-85.70260626299203, 38.14657969155578]","[-85.70169401763651, 38.14689512558871]"
...,...,...,...,...,...,...,...,...,...,...,...,...
31463,GLENVIEW AVE,2525,2566,GRAYSON CT,2626,2679,CABIN WAY,12896,1406,LOCAL,"[-85.64079071976235, 38.29284954144796]","[-85.64125447939766, 38.293433275538995]"
32265,COFFEE TREE PL,2101,2084,DEAD END,8594,9811,COFFEE TREE LN,2053,2053,LOCAL,"[-85.53795934713116, 38.181908231935815]","[-85.53694581749606, 38.18196699123739]"
32380,GARRS LN,2460,2488,LISA AVE,3661,3890,MILL CREEK DR,4085,4384,LOCAL,"[-85.81318601529698, 38.189609592766914]","[-85.81506403880756, 38.18972105387296]"


In [ ]:
# # # This is where Rehl Rd intersects Tucker station

""" SIFIDLOW	SIFIDHI	next	previouos
7109	5908	5908	NaN	NaN # next previous should be 12697?
9233	5908	12697	NaN	NaN
...
52804	12697	4674	NaN	NaN

This is where Rehl Rd intersects Tucker station

       | Tucker Station
       |
--Rehl--====Rehl====----Rehl--
                  |
                  | Tucker Station

This makes it too annoying to connect the segments using only SIFIDS. Lots of unusual cases.
We have exact geometry data, just use that
code for this already in centerline_data notebook """

' SIFIDLOW\tSIFIDHI\tnext\tpreviouos\n7109\t5908\t5908\tNaN\tNaN # next previous should be 12697?\n9233\t5908\t12697\tNaN\tNaN\n...\n52804\t12697\t4674\tNaN\tNaN\n\nThis is where Rehl Rd intersects Tucker station\n\n       | Tucker Station\n       |\n--Rehl--====Rehl====----Rehl--\n                  |\n                  | Tucker Station\n\nThis makes it too annoying to connect the segments using only SIFIDS. Lots of unusual cases.\nWe have exact geometry data, just use that\ncode for this already in centerline_data notebook '